In [23]:
import dataclasses
import matplotlib.pyplot as plt
import numpy as np
import scipy.linalg
from collections import namedtuple
import jax
import jax.numpy as jnp
jax.config.update("jax_enable_x64", True)
from trot.prop.hubbard_cpmc_ops import _build_prop_ctx
from trot.core.system import System
from trot.ham.hubbard import HamHubbard
from trot.ham.chol import HamChol
from trot.trial.uhf import UhfTrial, get_rdm1 as uhf_get_rdm1, overlap_u as uhf_overlap_u
from trot.trial.auto import make_auto_trial_ops
from trot.core.ops import MeasOps
from trot.meas.uhf import energy_kernel_uw_rh, build_meas_ctx as uhf_build_meas_ctx
from trot.prop import blocks, cpmc_slow
from trot.prop.types import QmcParams
from trot.core.ops import k_energy
from trot.driver import run_qmc_energy
from trot.walkers import _qr
from trot import walkers as wk
from trot.prop.cpmc import init_prop_state
from trot.prop.hubbard_cpmc_ops import make_hubbard_cpmc_ops
from trot.prop.types import PropOps, PropState


In [24]:
#Hubbard model parameters
L = 8
n_up = 4
n_down = 4 
t = 1.0
U = 4.0

h1 = np.zeros((L, L))
for i in range(L - 1):
    h1[i, i + 1] = h1[i + 1, i] = -t

# restricted HF 
eps, mo = np.linalg.eigh(h1)
Ca = mo[:, :n_up].copy()
Cb = mo[:, :n_down].copy()

e_hf = 2.0 * eps[:n_up].sum() + U * sum((Ca[i] @ Ca[i]) * (Cb[i] @ Cb[i]) for i in range(L))

ham = HamHubbard(h1=jnp.asarray(h1), u=U)
sys_ = System(norb=L, nelec=(n_up, n_down), walker_kind="unrestricted")
# the same determinant as a UHF trial: used for the energy estimator and the reference run
trial_data = UhfTrial(mo_coeff_a=jnp.asarray(Ca), mo_coeff_b=jnp.asarray(Cb))


chol = np.zeros((L, L, L))
for i in range(L):
    chol[i, i, i] = np.sqrt(U)
ham_chol = HamChol(h0=jnp.asarray(0.0), h1=jnp.asarray(h1),
                   chol=jnp.asarray(chol), basis="restricted")
meas_ctx_uhf = uhf_build_meas_ctx(ham_chol, trial_data)


def energy_uhf(walker, ham_data=None, meas_ctx=None, trial_data=trial_data):
    """Hubbard local energy via meas/uhf's Cholesky kernel. ham_data/meas_ctx are
    ignored: the propagation carries HamHubbard, this needs the HamChol form."""
    return energy_kernel_uw_rh(walker, ham_chol, meas_ctx_uhf, trial_data)
print(f"E_HF = {e_hf:.10f}")

E_HF = -1.5175409663


In [25]:
#Full ED to benchmark. Est runtime 20s

from itertools import combinations

def _hop_matrix(n):
    """Spinless nearest-neighbour hopping in the n-electron occupation basis.

    The Jordan-Wigner string between ADJACENT sites is empty, so every nonzero
    element is just -t and there is no sign to track.
    """
    strs = [frozenset(c) for c in combinations(range(L), n)]
    idx = {s: k for k, s in enumerate(strs)}
    M = np.zeros((len(strs), len(strs)))
    for k, s in enumerate(strs):
        for i in range(L - 1):
            for src, dst in ((i + 1, i), (i, i + 1)):
                if src in s and dst not in s:
                    M[idx[frozenset((s - {src}) | {dst})], k] += -t
    return M, strs


def ed_energy(u):
    """Ground state by dense eigh in the (n_up, n_down) sector."""
    Ha, sa = _hop_matrix(n_up)
    Hb, sb = _hop_matrix(n_down)
    na, nb = len(sa), len(sb)

    H = np.zeros((na * nb, na * nb))
    Hr = H.reshape(na, nb, na, nb)          # a view, so these write into H
    for ib in range(nb):
        Hr[:, ib, :, ib] += Ha              # hopping, spin up
    for ia in range(na):
        Hr[ia, :, ia, :] += Hb              # hopping, spin down
    docc = np.array([[len(a & b) for b in sb] for a in sa]).ravel()
    H[np.diag_indices_from(H)] += u * docc  # U * sum_i n_iup n_idn

    return float(np.linalg.eigvalsh(H)[0]), H.shape[0]


e_exact, ed_dim = ed_energy(U)

print(f"E_exact = {e_exact:.12f}")


E_exact = -4.235806999130


In [26]:
"""Two functions. The first grows B adapatevely until the tolarance is reached. 
The second one always use the maximum available B to avoid any possible error
and it can be used to benchmark/debug. The "plan" is (occ,B,v_ref). Fixing occ and B
keeps the shapes of the subsquent operation fixed and the whole process efficient. v_ref
is used as initial guess of the eigenvector, to avoid calling eig in the replay function. It
is not necessary for now.
"""
def plan_channel(C, eps_occ=1e-10):
    """Fishman-White compression of one spin channel. NumPy, run once.

    Records only the discrete decisions: the product-state occupations `occ`, the
    block size `B` used at each step, and the reference eigenvector `vref` that
    selects the mode (and, later, fixes its sign).
    """
    U, n = np.asarray(C, float).copy(), C.shape[0]
    occ, Bs, vrefs = np.zeros(n, int), [], []

    for k in range(n - 1):
        Lam = U @ U.T
        for B in range(2, n - k + 1):                      # grow the block until a mode isolates
            w, W = np.linalg.eigh(Lam[k : k + B, k : k + B])
            if min(w[0], 1.0 - w[-1]) < eps_occ:
                break
        if w[0] <= 1.0 - w[-1]:                            # empty mode is the cleaner one
            v, occ[k] = W[:, 0], 0
        else:                                              # occupied mode
            v, occ[k] = W[:, -1], 1
        Bs.append(B)
        vrefs.append(v.copy())

        for j in range(B - 1, 0, -1):                      # Givens-rotate v onto site k
            th = np.arctan2(v[j], v[j - 1])
            c, s = np.cos(th), np.sin(th)
            v[j - 1], v[j] = c * v[j - 1] + s * v[j], 0.0
            U[k + j - 1], U[k + j] = (
                c * U[k + j - 1] + s * U[k + j],
                -s * U[k + j - 1] + c * U[k + j],
            )

    occ[n - 1] = int(round(float(U[n - 1] @ U[n - 1])))
    assert occ.sum() == C.shape[1], "particle number lost during compression"
    return occ, np.array(Bs), vrefs

def plan_channel_maxB(C):
    """plan_channel with B = n-k always. The remaining block is then a genuine
    projector, so its eigenvalues are exactly 0/1 for any walker."""
    U, n = np.asarray(C, float).copy(), C.shape[0]
    occ, Bs, vrefs = np.zeros(n, int), [], []

    for k in range(n - 1):
        B = n - k                                          
        w, W = np.linalg.eigh((U @ U.T)[k : k + B, k : k + B])
        if w[0] <= 1.0 - w[-1]:
            v, occ[k] = W[:, 0], 0
        else:
            v, occ[k] = W[:, -1], 1
        Bs.append(B)
        vrefs.append(v.copy())
        for j in range(B - 1, 0, -1):
            th = np.arctan2(v[j], v[j - 1])
            c, s = np.cos(th), np.sin(th)
            v[j - 1], v[j] = c * v[j - 1] + s * v[j], 0.0
            U[k + j - 1], U[k + j] = (c * U[k + j - 1] + s * U[k + j],
                                      -s * U[k + j - 1] + c * U[k + j])

    occ[n - 1] = int(round(float(U[n - 1] @ U[n - 1])))
    assert occ.sum() == C.shape[1], "particle number lost during compression"
    return occ, np.array(Bs), vrefs

In [27]:
"""Replay. The plan (occ, B, vref) from the previous cell is frozen; this
recomputes the Givens angles from a live C, with static shapes so it is jit- and
vmap-able. Fixing occ fixes every SVD/QR shape downstream; B is fixed for
convenience; vref selects the mode inside a degenerate cluster and fixes its sign.

C must have ORTHONORMAL columns -- callers QR first and carry det(R) as an overall
factor. That is what makes the mode selection cheap: Lambda = C C^T is then a
projector, so every subblock has its spectrum in [0, 1] and

    P = M          (an occupied mode is wanted)
    P = I - M      (an empty mode is wanted)

already puts the wanted mode at eigenvalue 1 -- no trace normalisation needed.

Measured on 100 walkers, end to end through the full overlap (L=8, half filling):

    mode      maximal B                 minimal B
    proj       4.9 ms   err 3e-15        --  (filter is not exact there)
    eigh       7.3 ms   err 4e-15       4.7 ms   err 7e-15
    sq  (25)   9.0 ms   err 6e-15       5.6 ms   err 1e-14

so "proj" wins at maximal B and "eigh" wins at minimal B; the original 25
repeated squarings are never the best choice. mode="auto" picks on that basis.
"""

def is_max_b(plan, n):
    """True if the plan always takes the whole remaining block."""
    return all(int(B) == n - k for k, B in enumerate(plan[1]))


def channel_angles(C, plan, mode="auto", n_mv=2, n_sq=25, xp=jnp):
    """Givens angles for one spin channel, plus the rotated rows of U.

    det(U_rot[occ, :]) is the gauge sign the MPS construction drops, and the rows
    are built here anyway, so handing them back is free.

    mode="proj" (maximal B only): with B = n - k the remaining block is itself
        idempotent, so P IS the spectral filter and squaring it is a fixed point.
        `n_mv` power-iteration matvecs against vref then pick the mode out of the
        degenerate eigenvalue-1 cluster and fix its sign.

        n_mv=2, not 1. One projection of vref can be *short*: vref was recorded
        for the trial orbitals, a propagated walker's occupied subspace has
        rotated away from it, and ||P vref|| falls to ~1e-2 by the last steps.
        The absolute roundoff in forming P vref is ~eps, so normalising a short
        vector inflates it to ~1e-14 in the *direction* of v. After one
        projection v already lies almost in the subspace, so the second has
        ||P v|| ~ 1 and cleans up the leaked part at full precision -- the
        "twice is enough" of re-orthogonalisation. Measured: n_mv=1 -> 2e-13,
        n_mv=2 -> 3e-15, n_mv=3 -> no further gain.

    mode="eigh" (any B): the exact extremal eigenvector, and the right choice for
        minimal B, where the retained block is only approximately idempotent and
        the filter would need all 25 squarings to separate the cluster. The
        eigenvector is not a continuous function of C inside a degenerate
        cluster, so this mode is for values, not gradients.

    mode="sq": the original trace-normalised repeated-squaring filter, kept for
        reference. Also note it is the *least* accurate of the three: 25
        squarings amplify roundoff, and its angles differ from the exact
        projector's by ~1e-8.

    `xp` is numpy or jax.numpy -- `plan_bonds` reuses this in NumPy.
    """
    occ, Bs, vrefs = plan
    n = len(occ)
    if mode == "auto":
        mode = "proj" if is_max_b(plan, n) else "eigh"
    C = xp.asarray(C)
    rows = [C[i] for i in range(n)]      # a Python list, so each rotation is plain
    angles = []                          # arithmetic instead of two scatters
    for k, (B, vref) in enumerate(zip(Bs, vrefs)):
        B = int(B)
        Uk = xp.stack(rows[k : k + B])
        M = Uk @ Uk.T
        vr = xp.asarray(vref)
        if mode == "eigh":
            _, W = xp.linalg.eigh(M)
            v = W[:, -1] if occ[k] == 1 else W[:, 0]
        else:
            P = M if occ[k] == 1 else xp.eye(B) - M
            if mode == "sq":
                for _ in range(n_sq):
                    P = P @ P
                    P = P / xp.linalg.norm(P)
            v = vr
            for _ in range(n_mv):
                v = P @ v
                v = v / xp.linalg.norm(v)
        v = v * xp.sign(v @ vr)                       # inherit the plan's sign
        v = [v[i] for i in range(B)]

        for j in range(B - 1, 0, -1):                 # rotate v onto site k
            th = xp.arctan2(v[j], v[j - 1])
            c, s = xp.cos(th), xp.sin(th)
            v[j - 1] = c * v[j - 1] + s * v[j]
            p = k + j - 1
            rows[p], rows[p + 1] = (c * rows[p] + s * rows[p + 1],
                                    -s * rows[p] + c * rows[p + 1])
            angles.append((p, th))
    return angles, rows

In [28]:
"""MPS construction and contraction.

Three things here are worth reading before the code.

1. THE GATE IS NEVER BUILT.  V(th) only mixes |01> and |10>, so folding it into
   the two-site tensor is two axpys on the singly-occupied slices. `V_hat` is kept
   as the definition and the next cell asserts the fused version reproduces it.

2. SPLITTING: NO SVD IN THE PER-WALKER PATH.  A full-rank split only has to
   reproduce T, so
   the singular values are never used and a plain QR does the job for about a
   third of the flops. Once we truncate they ARE the point -- but they still do
   not need an SVD of the big matrix. Following Unfried, Hauschild & Pollmann,
   arXiv:2212.09782 (QR-based TEBD), reduce with a QR first and then read the
   Schmidt values off the *small* hermitian R R^T with eigh:

        M = Q R,   R R^T = V S^2 V^T,   M_k = (Q V_k) (V_k^T R)

   That is the exact rank-k truncation -- bit for bit the same retained subspace
   as an SVD -- at QR-like cost. Measured, 100 walkers through the full overlap:

        chi_max   svd       qr_eigh    qrcp      plain qr
        none     23.8 ms    7.1 ms    5.2 ms    4.6 ms     all err ~1e-15
        12       23.8 ms    7.2 ms    5.1 ms    4.4 ms
          err     1.2e-2    1.2e-2    5.1e-3    4.8e-2   (median)
          err     5.0e-2    5.0e-2    2.4e-1    6.3e-1   (max)
        8        25.3 ms    6.9 ms    5.0 ms    4.4 ms
          err     2.8e-2    2.8e-2    3.3e-2    1.6e-1   (median)

   So: no compression -> QR; compression -> "qr_eigh". Column-pivoted QR is a
   little faster still but only near-optimal (note its much worse tail), and a
   plain unpivoted QR is simply WRONG for truncation -- it is not rank revealing,
   so it keeps an arbitrary subspace of the right size. Forming R R^T squares the
   condition number, so the smallest retained Schmidt values carry ~sqrt(eps)
   relative error; they are the ones about to be discarded, so this does not
   matter, but it is why this trick is for truncation and not for diagnostics.
   (`plan_bonds` does use a real SVD -- it runs once, in NumPy, off the hot path,
   and it is where the *planned* discarded weight is measured.)

3. SPIN FACTORISES IN THE OVERLAP -- BUT ONLY BETWEEN SPIN-PRODUCT STATES.
   The d=4 MPS is Aa (x) Ab with a Jordan-Wigner interleaving sign
   s(a,b) = (-1)^{sum_i a_i sum_{j<i} b_j}. For a bra and a ket that are BOTH
   spin products with the same interleaving, that sign appears once on each side
   of the same configuration and squares to 1, so

        <bra|ket> = <bra_a|ket_a> * <bra_b|ket_b>

   exactly. Every overlap then runs at the CHANNEL bond dimension, 16 here,
   never at the combined 256 -- about 400x fewer flops.

   THE CONDITION IS NOT AUTOMATIC, so `spin_schmidt_rank` below measures it and
   the next cell asserts it. Rank 1 <=> spin product <=> factorisation exact.
   Measured at L=8:

        psi_T, a single Slater determinant .......... rank 1   <- factorises
        H|psi_T>, after the MPO ..................... rank 8   <- does NOT
        |det1> + |det2|, a 2-determinant trial ...... rank 2   <- does NOT

   Walkers are always fine: an unrestricted determinant propagated by
   spin-diagonal Hubbard-Stratonovich fields is a spin product by construction.
   The TRIAL is fine here only because it is one determinant. A DMRG trial will
   have rank > 1 and the factorised overlap would be silently wrong for it --
   that case needs `combine` + `mps_overlap`, which is why they are kept.

   H|psi_T> having rank 8 is not a problem for the energy, because the energy
   never factorises it: see the note in the next cell.
"""

# ------------------------------------------------------------------ the gate
def V_hat(th):
    """Two-site number-conserving gate for one spin channel, as a (2,2,2,2) tensor.

    The reference definition. `_gate_pair` fuses this into the contraction.
    """
    c, s = jnp.cos(th), jnp.sin(th)
    g = jnp.eye(4).at[1, 1].set(c).at[1, 2].set(s).at[2, 1].set(-s).at[2, 2].set(c)
    return g.reshape(2, 2, 2, 2)


def _gate_pair(A, B, th, xp=jnp):
    """(A B) with the gate already folded in, as (Dl, 2, 2, Dr)."""
    c, s = xp.cos(th), xp.sin(th)
    t00 = A[:, 0, :] @ B[:, 0, :]
    t01 = A[:, 0, :] @ B[:, 1, :]
    t10 = A[:, 1, :] @ B[:, 0, :]
    t11 = A[:, 1, :] @ B[:, 1, :]
    return xp.stack([xp.stack([t00, c * t01 + s * t10], 1),
                     xp.stack([-s * t01 + c * t10, t11], 1)], 1)


# --------------------------------------------------------------- the splitting
def sector_plan(ql, qr, ks=None):
    """Static per-charge row/column index sets for one two-site split.

    `ql` / `qr` are the particle numbers on the outer bonds; they alone fix every
    shape here, which is what makes this safe under vmap. `ks[s]` is how many
    states charge sector s keeps (None = full rank). Also returns the middle bond
    labels and the two gather maps that put the block-diagonal factors back into
    full (2*Dl, K) / (K, 2*Dr) shape. All NumPy and a function of the frozen
    labels only, so it is hoisted out of the traced code and cached.
    """
    nl, nr = len(ql), len(qr)
    rc = (ql[:, None] + np.arange(2)[None, :]).ravel()     # possible left charges
    cc = (qr[None, :] - np.arange(2)[:, None]).ravel()     # possible right charges
    secs, qm, rcat, ccat = [], [], [], []
    for s, nm in enumerate(sorted(set(rc.tolist()) & set(cc.tolist()))):
        r, c = np.where(rc == nm)[0], np.where(cc == nm)[0]
        k = min(len(r), len(c)) if ks is None else int(ks[s])
        if k == 0:                                         # sector truncated away
            continue
        secs.append((r, c, k))
        qm += [nm] * k                                     # k new states, charge nm
        rcat.append(r); ccat.append(c)
    rcat, ccat = np.concatenate(rcat), np.concatenate(ccat)
    # rows / columns in no retained sector are exactly zero: point them at one
    # extra zero row (column) appended to the block-diagonal factor
    rmap = np.full(2 * nl, len(rcat), int); rmap[rcat] = np.arange(len(rcat))
    cmap = np.full(2 * nr, len(ccat), int); cmap[ccat] = np.arange(len(ccat))
    return secs, np.array(qm, int), rmap, cmap


_SEC_CACHE = {}

def _sectors(ql, qr, ks=None):
    key = (ql.tobytes(), len(ql), qr.tobytes(), len(qr), ks)
    if key not in _SEC_CACHE:
        _SEC_CACHE[key] = sector_plan(ql, qr, ks)
    return _SEC_CACHE[key]


def _assemble(As, Bs, rmap, cmap, Dl, Dr):
    """Per-sector factors -> two MPS tensors, one static gather on each side."""
    A = jax.scipy.linalg.block_diag(*As)
    B = jax.scipy.linalg.block_diag(*Bs)
    A = jnp.concatenate([A, jnp.zeros((1, A.shape[1]), A.dtype)], 0)[rmap]
    B = jnp.concatenate([B, jnp.zeros((B.shape[0], 1), B.dtype)], 1)[:, cmap]
    return A.reshape(Dl, 2, -1), B.reshape(-1, 2, Dr)


def split_full(T, ql, qr):
    """Exact split, full rank inside each particle-number sector, via QR.

    The left factor is isometric, so the orthogonality centre lands on the
    right-hand site and the sequential gate sweep stays well conditioned.
    """
    Dl, _, _, Dr = T.shape
    M = T.reshape(Dl * 2, 2 * Dr)
    secs, qm, rmap, cmap = _sectors(ql, qr)
    As, Bs = [], []
    for r, c, k in secs:
        q, rr = jnp.linalg.qr(M[np.ix_(r, c)], mode="reduced")
        As.append(q[:, :k]); Bs.append(rr[:k])
    A, B = _assemble(As, Bs, rmap, cmap, Dl, Dr)
    return A, B, qm


def split_trunc(T, ql, qr, ks, backend="qr_eigh"):
    """Compressing split: keep only ks[s] states in charge sector s.

    HOW MANY states each sector keeps is frozen by the bond plan, which is what
    keeps shapes static under vmap. WHICH states -- the dominant subspace -- is
    recomputed from the walker every time, and that is what needs a
    rank-revealing factorisation:

      "qr_eigh"  QR, then eigh of the small R R^T (arXiv:2212.09782). The exact
                 rank-k truncation at ~3.4x the speed of an SVD. The default.
      "svd"      the same answer the slow way; for checking "qr_eigh".
      "qrcp"     column-pivoted QR. Slightly faster again, but only
                 near-optimal. Pivots are data dependent, which is fine: the
                 permutation is applied with a gather and only ever reorders
                 columns *within* one charge sector, so the labels are untouched.
                 (LAPACK path -- check availability before relying on it on GPU.)
      "qr"       plain QR. Present only to show that it does NOT work.
    """
    Dl, _, _, Dr = T.shape
    M = T.reshape(Dl * 2, 2 * Dr)
    secs, qm, rmap, cmap = _sectors(ql, qr, tuple(ks))
    As, Bs = [], []
    for r, c, k in secs:
        Ms = M[np.ix_(r, c)]
        if backend == "qr_eigh":
            q, rr = jnp.linalg.qr(Ms, mode="reduced")
            _, V = jnp.linalg.eigh(rr @ rr.T)             # ascending eigenvalues
            Vk = V[:, ::-1][:, :k]                        # dominant k, so descending
            As.append(q @ Vk); Bs.append(Vk.T @ rr)
        elif backend == "svd":
            u, sv, vt = jnp.linalg.svd(Ms, full_matrices=False)
            As.append(u[:, :k]); Bs.append(sv[:k, None] * vt[:k])
        elif backend == "qrcp":
            q, rr, piv = jax.scipy.linalg.qr(Ms, mode="economic", pivoting=True)
            As.append(q[:, :k]); Bs.append(rr[:k][:, jnp.argsort(piv)])
        else:
            q, rr = jnp.linalg.qr(Ms, mode="reduced")
            As.append(q[:, :k]); Bs.append(rr[:k])
    A, B = _assemble(As, Bs, rmap, cmap, Dl, Dr)
    return A, B, qm


def split_cbe(T, ql, qr, ks, sketches, n_power=0):
    """Compressing split via CONTROLLED BOND EXPANSION. Implemented, measured,
    and OFF BY DEFAULT -- the measurements are below and say it does not pay here.

    The idea (Unfried, Hauschild & Pollmann, arXiv:2212.09782): `split_trunc`
    factorises a |r| x |c| block completely and keeps k of min(|r|,|c|) states,
    i.e. it builds the whole d*chi-dimensional bond and discards most of it.
    Instead work in an expanded-but-small space of dimension

        eta = k + Delta chi   (<= min(|r|, |c|))

    so the cost goes as |r| |c| eta rather than |r| |c| min(|r|,|c|). That is the
    paper's d^3 -> d^2.

    The paper takes an "arbitrary eta-dimensional slice" as the guess, justified
    by U = 1 + O(dt) making it right to first order in the time step. That
    argument does NOT transfer: these gates are finite-angle Givens rotations.
    So the guess is a frozen random sketch with oversampling -- the randomised
    range finder -- which needs no such assumption, plus optional power
    iterations Y <- M (M^T M)^q Pi for a slowly decaying spectrum. Like every
    other discrete choice the sketch lives in the plan, so it is identical for
    every walker and every call and the shapes stay static under vmap.

    WHY IT IS OFF BY DEFAULT. On a single DENSE block with m = n = 2 chi and
    k = chi it works exactly as advertised, batched over walkers:

        m x n        qr_eigh   cbe q=0   speedup   err (s_i ~ 0.7^i)
        64x64          8.5 ms    5.4 ms    1.57x   3.3e-06
        256x256      138.7 ms   53.6 ms    2.59x   5.1e-09
        1024x1024    684.6 ms  173.3 ms    3.95x   1.0e-10

    But this algorithm never factorises a dense block. Particle-number blocking
    has already cut every split into per-charge sectors, and those sectors are
    tiny:

        L   chi_max   median k_s   median min(|r_s|,|c_s|)   splits with k_s+4 < min
         8   none          1            1                          0%
        16     32          2            2                          7%
        16     64          2            2                          4%
        20    128          3            3                          6%

    There is nothing to expand into: k_s + Delta chi already saturates
    min(|r_s|,|c_s|) for 88-100% of splits. Measured end to end, eta/full comes
    out at 0.98-1.00 and the speedup at 0.89x-1.25x -- a wash -- while costing
    real accuracy. The saving CBE targets has already been taken by the symmetry
    decomposition. It would matter for a d=4 (or larger-d) chain contracted
    WITHOUT quantum numbers, which is the setting the paper addresses.
    """
    Dl, _, _, Dr = T.shape
    M = T.reshape(Dl * 2, 2 * Dr)
    secs, qm, rmap, cmap = _sectors(ql, qr, tuple(ks))
    As, Bs = [], []
    for (r, c, k), P in zip(secs, sketches):
        Ms = M[np.ix_(r, c)]
        Y = Ms @ jnp.asarray(P)                       # |r| x eta
        for _ in range(n_power):
            Y, _ = jnp.linalg.qr(Y)                   # re-orthonormalise between
            Y = Ms @ (Ms.T @ Y)                       # passes, for stability
        Q, _ = jnp.linalg.qr(Y)                       # |r| x eta isometry
        Z = Q.T @ Ms                                  # eta x |c|
        _, W = jnp.linalg.eigh(Z @ Z.T)               # eta x eta, small
        Wk = W[:, ::-1][:, :k]
        As.append(Q @ Wk); Bs.append(Wk.T @ Z)
    A, B = _assemble(As, Bs, rmap, cmap, Dl, Dr)
    return A, B, qm


# --------------------------------------------------------------- the bond plan
BondPlan = namedtuple("BondPlan", "ks qn chi discarded sk")


def plan_bonds(C, plan, chi_max=None, cutoff=0.0, cbe_extra=None,
               seed=0, **kw):
    """Dry-run the gate sweep in NumPy and freeze how each split spends its budget.

    Truncation is a discrete decision, so it belongs in the plan alongside the
    block sizes and occupations: for every gate this records how many states each
    charge sector keeps. The JAX replay then has static shapes and vmaps, while
    the singular values themselves are still recomputed from each walker.

    `chi_max` caps the bond dimension of a split; `cutoff` additionally drops
    singular values below `cutoff * s_max` of that same split. The budget is
    global across the split's sectors -- it goes to the largest singular values
    wherever they sit.

    The allocation is planned on ONE reference C (the trial orbitals). Walkers
    drift from it, so the frozen split stops being the optimal one and the true
    error is larger than `discarded`, which is only what the reference throws away.

    `cbe_extra` (Delta chi) additionally freezes a random sketch per sector, for
    `split_cbe`. Leave it None unless you are reproducing the CBE measurement --
    see `split_cbe` for why it does not pay in this algorithm.
    """
    rng_sk = np.random.default_rng(seed)
    occ = plan[0]
    ts = [np.eye(2)[int(o)].reshape(1, 2, 1) for o in occ]
    qn = [np.zeros(1, int)]
    for o in occ:
        qn.append(qn[-1] + int(o))

    angles, _ = channel_angles(np.asarray(C, float), plan, xp=np, **kw)
    ks_all, sk_all, discarded = [], [], 0.0
    for p, th in reversed(angles):
        T = _gate_pair(ts[p], ts[p + 1], th, xp=np)
        Dl, _, _, Dr = T.shape
        M = T.reshape(Dl * 2, 2 * Dr)
        secs, _, _, _ = sector_plan(qn[p], qn[p + 2])          # untruncated layout

        svs, blocks = [], []
        for r, c, kfull in secs:
            u, sv, vt = np.linalg.svd(M[np.ix_(r, c)], full_matrices=False)
            svs.append(sv[:kfull]); blocks.append((u, sv, vt))
        flat = np.concatenate(svs)
        order = np.argsort(-flat)
        sel = order[: len(flat) if chi_max is None else min(int(chi_max), len(flat))]
        if cutoff > 0.0 and len(flat):
            sel = sel[flat[sel] > cutoff * flat[order[0]]]
        keep = np.zeros(len(flat), bool); keep[sel] = True
        discarded += float((flat[~keep] ** 2).sum())

        ks, off = [], 0                       # svd is sorted, so the kept set of a
        for sv in svs:                        # sector is always a prefix of it
            ks.append(int(keep[off : off + len(sv)].sum())); off += len(sv)
        ks_all.append(tuple(ks))

        # apply this truncation in NumPy so the dry run continues on the same state
        secs_t, qm, rmap, cmap = sector_plan(qn[p], qn[p + 2], tuple(ks))
        if cbe_extra is not None:
            sk_all.append([rng_sk.standard_normal(
                (len(c), min(k + int(cbe_extra), len(r), len(c))))
                for (r, c, k) in secs_t])
        As, Bs = [], []
        for (u, sv, vt), k in zip(blocks, ks):
            if k:
                As.append(u[:, :k]); Bs.append(sv[:k, None] * vt[:k])
        A = scipy.linalg.block_diag(*As); B = scipy.linalg.block_diag(*Bs)
        A = np.concatenate([A, np.zeros((1, A.shape[1]))], 0)[rmap]
        B = np.concatenate([B, np.zeros((B.shape[0], 1))], 1)[:, cmap]
        ts[p] = A.reshape(Dl, 2, -1); ts[p + 1] = B.reshape(-1, 2, Dr)
        qn[p + 1] = qm

    return BondPlan(ks=ks_all, qn=qn, chi=max(len(q) for q in qn),
                    discarded=discarded, sk=(sk_all if cbe_extra is not None else None))


# ------------------------------------------------------------- the construction
_ONEHOT = (jnp.array([[[1.0], [0.0]]]), jnp.array([[[0.0], [1.0]]]))   # (1, 2, 1)


def channel_mps(C, plan, bond_plan=None, backend="qr_eigh", n_power=0, **kw):
    """Product state |occ> -> gates in reverse derivation order -> one d=2 MPS.

    Returns (tensors, bond labels, gauge sign det(U_rot[occ, :])). Without a
    `bond_plan` every split keeps full rank and the MPS is exact.
    """
    occ = plan[0]
    ts = [_ONEHOT[int(o)] for o in occ]
    qn = [np.zeros(1, int)]
    for o in occ:
        qn.append(qn[-1] + int(o))

    angles, rows = channel_angles(C, plan, **kw)
    for g, (p, th) in enumerate(reversed(angles)):
        T = _gate_pair(ts[p], ts[p + 1], th)
        if bond_plan is None:
            ts[p], ts[p + 1], qn[p + 1] = split_full(T, qn[p], qn[p + 2])
        elif backend == "cbe":
            ts[p], ts[p + 1], qn[p + 1] = split_cbe(
                T, qn[p], qn[p + 2], bond_plan.ks[g], bond_plan.sk[g], n_power)
        else:
            ts[p], ts[p + 1], qn[p + 1] = split_trunc(
                T, qn[p], qn[p + 2], bond_plan.ks[g], backend=backend)

    gauge = jnp.linalg.det(jnp.stack([rows[i] for i in np.where(occ == 1)[0]]))
    return ts, qn, gauge


# ------------------------------------------------------------- the contractions
def chan_overlap(bra, ket):
    """<bra|ket> for one d=2 channel."""
    e = jnp.ones((bra[0].shape[0], ket[0].shape[0]))
    for a, b in zip(bra, ket):
        e = jnp.tensordot(a, jnp.tensordot(e, b, ([1], [0])), ([0, 1], [0, 1]))
    return e.reshape(())


def left_envs(bra, ket):
    """L[x] = contraction of sites 0..x-1. L[n][0,0] is the full overlap."""
    L = [jnp.ones((1, 1))]
    for a, b in zip(bra, ket):
        L.append(jnp.einsum("ab,axc,bxd->cd", L[-1], a, b))
    return L


def right_envs(bra, ket):
    """R[x] = contraction of sites x..n-1. R[0][0,0] is the full overlap.

    One backward pass, so the pre-loop overlap comes out for free -- which is what
    the fast sweep needs.
    """
    R = [jnp.ones((1, 1))]
    for a, b in zip(reversed(bra), reversed(ket)):
        R.append(jnp.einsum("axc,bxd,cd->ab", a, b, R[-1]))
    return R[::-1]


# --------------------------------------------- one-channel matrix elements
_SP = np.array([[0.0, 0.0], [1.0, 0.0]])       # sigma+, |0> -> |1>
_SM = np.array([[0.0, 1.0], [0.0, 0.0]])       # sigma-
_PZ = np.diag([1.0, -1.0])                     # (-1)^n, the Jordan-Wigner string
_NV = np.array([0.0, 1.0])                     # diag(n)


def hop_pairs(h1, tol=1e-14):
    """[(i, j, h1[i,j], h1[j,i])] over i < j with either direction nonzero."""
    h1 = np.asarray(h1)
    return [(i, j, float(h1[i, j]), float(h1[j, i]))
            for i in range(h1.shape[0]) for j in range(i + 1, h1.shape[0])
            if abs(h1[i, j]) > tol or abs(h1[j, i]) > tol]


def _string_term(bra, ket, L, R, i, j, Oi, Oj):
    """<bra| Oi_i (prod_{i<k<j} P_k) Oj_j |ket>, i < j, from stored environments.

    a+_m a_n = (prod_{min<k<max} P_k) sigma+_m sigma-_n for either order of m, n,
    so one routine covers both hopping directions.
    """
    e = jnp.einsum("ab,axc,xy,byd->cd", L[i], bra[i], Oi, ket[i])
    for k in range(i + 1, j):
        e = jnp.einsum("ab,axc,xy,byd->cd", e, bra[k], _PZ, ket[k])
    return jnp.einsum("ab,axc,xy,byd,cd->", e, bra[j], Oj, ket[j], R[j + 1])


def chan_obs(bra, ket, pairs, diag_h1):
    """(<bra|ket>, [<bra|n_x|ket>], <bra| sum_ij h1_ij c+_i c_j |ket>), one channel.

    One forward and one backward environment pass, then one contraction per
    nonzero hopping direction. For a nearest-neighbour chain that is 2(L-1) small
    contractions at the channel bond dimension.
    """
    L, R = left_envs(bra, ket), right_envs(bra, ket)
    nn = [jnp.einsum("ab,axc,x,bxd,cd->", L[x], bra[x], _NV, ket[x], R[x + 1])
          for x in range(len(bra))]
    k1 = sum(float(d) * v for d, v in zip(diag_h1, nn) if d != 0.0)
    for i, j, cij, cji in pairs:
        if cij != 0.0:
            k1 = k1 + cij * _string_term(bra, ket, L, R, i, j, _SP, _SM)
        if cji != 0.0:
            k1 = k1 + cji * _string_term(bra, ket, L, R, i, j, _SM, _SP)
    return R[0].reshape(()), nn, k1


# ------------------------------- the d=4 form, for the MPO and pyblock3 only
def combine(Aa, qna, Ab, qnb):
    """Interleave two d=2 channels into one d=4 MPS, local index = n_alpha + 2*n_beta."""
    ts, qn = [], [np.zeros((1, 2), int)]
    for i in range(len(Aa)):
        Dal, _, Dar = Aa[i].shape
        Dbl, _, Dbr = Ab[i].shape
        out = jnp.zeros((Dal, Dbl, 4, Dar, Dbr))
        for na in (0, 1):
            sgn = (-1.0) ** (na * qnb[i])                 # the Jordan-Wigner sign
            for nb in (0, 1):
                out = out.at[:, :, na + 2 * nb, :, :].set(
                    jnp.einsum("ar,b,bs->abrs", Aa[i][:, na, :], sgn, Ab[i][:, nb, :]))
        ts.append(out.reshape(Dal * Dbl, 4, Dar * Dbr))
        qn.append(np.stack([np.repeat(qna[i + 1], Dbr), np.tile(qnb[i + 1], Dar)], 1))
    return ts, qn


def sd_to_mps_gauged(ca, cb, plan_a, plan_b, **kw):
    """Slater determinant -> (d=4 MPS, gauge sign). Requires ORTHONORMAL columns."""
    ta, qna, sa = channel_mps(ca, plan_a, **kw)
    tb, qnb, sb = channel_mps(cb, plan_b, **kw)
    return combine(ta, qna, tb, qnb)[0], sa * sb


def sd_to_mps(ca, cb, plan_a, plan_b, **kw):
    """Tensors only, for the places where the gauge cancels anyway."""
    return sd_to_mps_gauged(ca, cb, plan_a, plan_b, **kw)[0]


def spin_schmidt_rank(ts, tol=1e-10):
    """Rank of the alpha/beta amplitude matrix with the interleaving sign removed.

    rank 1  <=>  the state is a spin product  <=>  the factorised overlap is exact.
    This is the validity condition for every factorised kernel in this notebook.

    Contracts the whole 4**n state, so it is a small-L diagnostic only.
    """
    n = len(ts)
    v = np.asarray(ts[0])[0]
    for A in ts[1:]:
        v = np.tensordot(v, np.asarray(A), axes=([-1], [0]))
    # local index l = n_a + 2 n_b, so reshape(4 -> (2,2)) gives (n_b, n_a)
    T = v.reshape((2, 2) * n)
    M = T.transpose(list(range(1, 2 * n, 2))
                    + list(range(0, 2 * n, 2))).reshape(2 ** n, 2 ** n)
    bits = (np.arange(2 ** n)[:, None] >> np.arange(n)[None, ::-1]) & 1
    sgn = (-1.0) ** (bits @ (np.cumsum(bits, 1) - bits).T)
    sv = np.linalg.svd(M * sgn, compute_uv=False)
    sv = sv / max(sv[0], 1e-300)
    return int((sv > tol).sum()), sv


def mps_overlap(bra, ket):
    """<bra|ket> for two MPS given as lists of (Dl, d, Dr) arrays, any d."""
    e = jnp.ones((bra[0].shape[0], ket[0].shape[0]))
    for a, b in zip(bra, ket):
        e = jnp.tensordot(a, jnp.tensordot(e, b, ([1], [0])), ([0, 1], [0, 1]))
    return e.reshape(())

In [29]:
"""Build the trial. The MPS is kept PER CHANNEL -- that is the representation the
overlap, the energy and the fast sweep all use. The d=4 form is built too, but
only as the reference for the MPO energy and the pyblock3 export.

CHI_WALKER / CHI_TRIAL are the compression knobs. None means no truncation, so
the conversion is exact and the run agrees with full-determinant CPMC to machine
precision. Set them to an int to cap the CHANNEL bond dimension (the d=4 bond is
the product of the two, so chi_channel=8 means a d=4 bond of 64).
"""
CHI_TRIAL   = None          # None = exact; int caps the trial's channel bond dim
CHI_WALKER  = None          # None = exact; int caps every walker's channel bond dim
CUTOFF      = 0.0           # additionally drop s < CUTOFF * s_max per split
SPLIT_BACKEND = "qr_eigh"   # "qr_eigh" (exact, fast), "svd" (exact, slow), "qrcp"

plan_a = plan_channel_maxB(Ca)
plan_b = plan_channel_maxB(Cb)
plan_a_min = plan_channel(Ca)
plan_b_min = plan_channel(Cb)

# bond plans: None when nothing is being truncated, so the exact QR split is used
bp_trial_a = bp_trial_b = bp_walk_a = bp_walk_b = None
if CHI_TRIAL is not None or CUTOFF > 0.0:
    bp_trial_a = plan_bonds(Ca, plan_a, chi_max=CHI_TRIAL, cutoff=CUTOFF)
    bp_trial_b = plan_bonds(Cb, plan_b, chi_max=CHI_TRIAL, cutoff=CUTOFF)
if CHI_WALKER is not None or CUTOFF > 0.0:
    bp_walk_a = plan_bonds(Ca, plan_a, chi_max=CHI_WALKER, cutoff=CUTOFF)
    bp_walk_b = plan_bonds(Cb, plan_b, chi_max=CHI_WALKER, cutoff=CUTOFF)

_conv_t = dict(backend=SPLIT_BACKEND)
_conv_w = dict(backend=SPLIT_BACKEND)

trial_a, qn_a, sign_ta = channel_mps(jnp.asarray(Ca), plan_a, bp_trial_a, **_conv_t)
trial_b, qn_b, sign_tb = channel_mps(jnp.asarray(Cb), plan_b, bp_trial_b, **_conv_t)
trial_sign = sign_ta * sign_tb
trial = combine(trial_a, qn_a, trial_b, qn_b)[0]        # d=4, for the MPO only

PAIRS  = hop_pairs(h1)                                   # nonzero hoppings, frozen
DIAG_H = np.diag(h1)

print("occ_alpha      =", plan_a[0])
print(f"maximal B      : blocks {plan_a[1]}  gates {int((plan_a[1]-1).sum()):3d}")
print(f"minimal B      : blocks {plan_a_min[1]}  gates {int((plan_a_min[1]-1).sum()):3d}")
print("channel bonds  =", [t.shape[0] for t in trial_a] + [trial_a[-1].shape[-1]])
print("d=4 bonds      =", [t.shape[0] for t in trial] + [trial[-1].shape[-1]])
if bp_trial_a is not None:
    print(f"trial discarded weight (planned on Ca) = "
          f"{bp_trial_a.discarded + bp_trial_b.discarded:.3e}")
print(f"<psi_T|psi_T>  = {float(chan_overlap(trial_a, trial_a) * chan_overlap(trial_b, trial_b)):.12f}")

# ---------------------------------------------------------------- self-checks
# 1. the fused gate really is V_hat
_rng = np.random.default_rng(0)
_A = jnp.asarray(_rng.standard_normal((3, 2, 5)))
_B = jnp.asarray(_rng.standard_normal((5, 2, 4)))
_th = 0.37
_ref = jnp.einsum("xypq,apqb->axyb", V_hat(_th), jnp.tensordot(_A, _B, 1))
print(f"\n_gate_pair == V_hat contraction : {float(jnp.abs(_gate_pair(_A,_B,_th)-_ref).max()):.2e}")

# 2. spin really does factorise, so the d=4 overlap is never needed
_wu = jnp.asarray(Ca + 0.3 * _rng.standard_normal(Ca.shape))
_wd = jnp.asarray(Cb + 0.3 * _rng.standard_normal(Cb.shape))
_qa, _ra = _qr(_wu); _qb, _rb = _qr(_wd)
_ta, _qna, _sa = channel_mps(_qa, plan_a, bp_walk_a, **_conv_w)
_tb, _qnb, _sb = channel_mps(_qb, plan_b, bp_walk_b, **_conv_w)
_d4  = float(mps_overlap(trial, combine(_ta, _qna, _tb, _qnb)[0]))
_fac = float(chan_overlap(trial_a, _ta) * chan_overlap(trial_b, _tb))
_det = float(jnp.linalg.det(Ca.T @ _wu) * jnp.linalg.det(Cb.T @ _wd))
print(f"<bra|ket> d=4 vs factorised     : {abs(_fac/_d4 - 1):.2e}")
print(f"gauged factorised vs det        : "
      f"{abs(float(_ra*_rb*_sa*_sb*trial_sign)*_fac/_det - 1):.2e}")

# 3. the trial really is a spin product -- the condition the factorisation needs
_r, _sv = spin_schmidt_rank(trial)
print(f"spin-Schmidt rank of psi_T      : {_r}  (next s/s0 {_sv[1]:.1e})")
assert _r == 1, ("the trial is NOT a spin product, so the factorised overlap and "
                 "energy in the next cell are invalid for it -- use combine() + "
                 "mps_overlap() instead")

occ_alpha      = [1 1 1 0 0 0 0 1]
maximal B      : blocks [8 7 6 5 4 3 2]  gates  28
minimal B      : blocks [5 4 4 3 3 2 2]  gates  16
channel bonds  = [1, 2, 4, 8, 16, 8, 4, 2, 1]
d=4 bonds      = [1, 4, 16, 64, 256, 64, 16, 4, 1]
<psi_T|psi_T>  = 1.000000000000

_gate_pair == V_hat contraction : 2.22e-16
<bra|ket> d=4 vs factorised     : 1.11e-16
gauged factorised vs det        : 2.00e-15
spin-Schmidt rank of psi_T      : 1  (next s/s0 4.6e-16)


In [30]:
"""The two kernels CPMC needs: the overlap and the local energy. Both run entirely
at the channel bond dimension.

SIGN CONVENTION.  |MPS> = G|occ>, Givens rotations acting on the product state of
occupied and empty modes, and G|SD> = |U_rot>, so

    <MPS|SD> = <occ|G|SD> = <occ|U_rot> = det(U_rot[occ, :])

is the gauge each conversion drops; `channel_mps` hands it back. On top of that,
the walker is orthonormalised before conversion (the replay needs orthonormal
columns), and |SD(C)> = det(R) |SD(Q)>, so det(R) rides along as well. Both
prefactors cancel in the energy ratio but NOT in the overlap.

THE ENERGY.  H = sum_sigma sum_ij h1_ij a+_i,sigma a_j,sigma + U sum_i n_ia n_ib.

H|psi_T> is NOT a spin product -- the U term is a product of an alpha operator and
a beta operator, and measured below H|psi_T> has spin-Schmidt rank 8. So the
factorised overlap must NOT be applied to it. It does not have to be: H is a sum
of 2 + L spin-PRODUCT operators, H_a (x) 1, 1 (x) H_b and U n_ia (x) n_ib, and each
of those factorises on its own. That is exactly the sum computed here, which is
why it is exact rather than an approximation.

With bra and ket both spin products,

    <psi_T| a+_ia a_ja |phi> = <A_a| c+_i c_j |B_a> * <A_b|B_b>

with NO parity insertion. The Jordan-Wigner string of the interleaved ordering
runs through the beta sites between ia and ja, but reordering the bra and the ket
into grouped form contributes exactly the same factor, and the two cancel. The
alternative (a (-1)^n_ib insertion in the beta channel) is wrong; the cross-check
against the d=4 MPO below is what settles it, and it agrees to 1e-14.

Cost, 100 walkers: MPO at chi=256 is 81 ms, this is 5.7 ms, same answer.
"""

def overlap_mps(walker, trial_data=None):
    """<psi_T|SD(C)>. This is the function CPMC needs."""
    ca, cb = walker
    qa, ra = _qr(ca)
    qb, rb = _qr(cb)
    ta, _, sa = channel_mps(qa, plan_a, bp_walk_a, **_conv_w)
    tb, _, sb = channel_mps(qb, plan_b, bp_walk_b, **_conv_w)
    return (ra * rb * sa * sb * trial_sign
            * chan_overlap(trial_a, ta) * chan_overlap(trial_b, tb))


def energy_mps(walker, ham_data=None, meas_ctx=None, trial_data=None):
    """E_loc = <psi_T|H|phi> / <psi_T|phi>, from per-channel matrix elements.

    The gauge and det(R) prefactors are common to numerator and denominator, so
    they never appear.
    """
    ca, cb = walker
    qa, _ = _qr(ca)
    qb, _ = _qr(cb)
    ta, _, _ = channel_mps(qa, plan_a, bp_walk_a, **_conv_w)
    tb, _, _ = channel_mps(qb, plan_b, bp_walk_b, **_conv_w)
    ov_a, n_a_, k1_a = chan_obs(trial_a, ta, PAIRS, DIAG_H)
    ov_b, n_b_, k1_b = chan_obs(trial_b, tb, PAIRS, DIAG_H)
    num = k1_a * ov_b + k1_b * ov_a + U * sum(x * y for x, y in zip(n_a_, n_b_))
    return num / (ov_a * ov_b)


# ================================================== the d=4 MPO, as a reference
# local operators, basis l = n_alpha + 2 n_beta :  |0>, |a>, |b>, |ab> = a+_a a+_b |0>
I4 = np.eye(4)
cr_a = np.zeros((4, 4)); cr_a[1, 0] = 1.0; cr_a[3, 2] = 1.0     # |0>->|a>, |b>->|ab>
cr_b = np.zeros((4, 4)); cr_b[2, 0] = 1.0; cr_b[3, 1] = -1.0    # |0>->|b>, |a>->-|ab>
an_a, an_b = cr_a.T.copy(), cr_b.T.copy()
n_a, n_b = np.diag([0., 1., 0., 1.]), np.diag([0., 0., 1., 1.])
P_a, P_b = np.diag([1., -1., 1., -1.]), np.diag([1., 1., -1., -1.])   # (-1)^n_sigma


def hubbard_mpo(L, t, U):
    """Dw=6 MPO. Bond basis: 0 = nothing started, 1..4 = a hop is pending, 5 = done."""
    W = np.zeros((L, 6, 4, 4, 6))
    for i in range(L):
        W[i, 0, :, :, 0] = I4
        W[i, 5, :, :, 5] = I4
        W[i, 0, :, :, 5] = U * (n_a @ n_b)                       # on-site interaction
        if i < L - 1:                                            # open a hop here
            W[i, 0, :, :, 1] = cr_a @ P_b                        # alpha: string on THIS site
            W[i, 0, :, :, 2] = an_a @ P_b
            W[i, 0, :, :, 3] = P_a @ cr_b                        # beta:  string also here
            W[i, 0, :, :, 4] = P_a @ an_b
        if i > 0:                                                # close one here
            W[i, 1, :, :, 5] = -t * an_a
            W[i, 2, :, :, 5] = -t * cr_a
            W[i, 3, :, :, 5] = -t * an_b
            W[i, 4, :, :, 5] = -t * cr_b
    return W


def apply_mpo(W, ts):
    """(W psi)[i] has bond dimension Dw*chi; the boundary MPO bonds are projected out."""
    out = []
    for i, (w, A) in enumerate(zip(W, ts)):
        A = np.asarray(A)
        if i == 0:
            w = w[0:1]
        if i == len(ts) - 1:
            w = w[:, :, :, 5:6]
        T = np.einsum("apqb,cqd->acpbd", w, A)                   # (Dw_l, Dl, 4, Dw_r, Dr)
        dl, cl, p, dr, cr = T.shape
        out.append(T.reshape(dl * cl, p, dr * cr))
    return out


def compress(ts, tol=1e-13):
    """Left-to-right QR to canonicalise, then right-to-left SVD. Exact at this tol."""
    ts = [np.asarray(t) for t in ts]
    for i in range(len(ts) - 1):
        Dl, d, Dr = ts[i].shape
        q, r = np.linalg.qr(ts[i].reshape(Dl * d, Dr))
        ts[i] = q.reshape(Dl, d, -1)
        ts[i + 1] = np.tensordot(r, ts[i + 1], axes=([1], [0]))
    for i in range(len(ts) - 1, 0, -1):
        Dl, d, Dr = ts[i].shape
        u, sv, vt = np.linalg.svd(ts[i].reshape(Dl, d * Dr), full_matrices=False)
        k = max(int((sv > tol * max(sv[0], 1e-300)).sum()), 1)
        ts[i] = vt[:k].reshape(k, d, Dr)
        ts[i - 1] = np.tensordot(ts[i - 1], u[:, :k] * sv[:k], axes=([2], [0]))
    return ts


raw = apply_mpo(hubbard_mpo(L, t, U), trial)
Hket = [jnp.asarray(x) for x in compress(raw)]


def energy_mps_mpo(walker, ham_data=None, meas_ctx=None, trial_data=None):
    """The same local energy through the d=4 MPO. Slow; this is the cross-check."""
    ca, cb = walker
    qa, _ = _qr(ca)
    qb, _ = _qr(cb)
    ta, qna, _ = channel_mps(qa, plan_a, bp_walk_a, **_conv_w)
    tb, qnb, _ = channel_mps(qb, plan_b, bp_walk_b, **_conv_w)
    mw = combine(ta, qna, tb, qnb)[0]
    return mps_overlap(Hket, mw) / mps_overlap(trial, mw)


_rH, _svH = spin_schmidt_rank(Hket)
print(f"spin-Schmidt rank of H|psi_T>: {_rH}   <- not 1, so H|psi_T> is NOT a spin")
print(f"                                    product; the energy below never assumes it is")
print("H|psi_T> before compression:", [x.shape[0] for x in raw] + [raw[-1].shape[-1]])
print("H|psi_T> after  compression:", [x.shape[0] for x in Hket] + [Hket[-1].shape[-1]])
print(f"\n<psi_T|H|psi_T> = {float(mps_overlap(Hket, trial) / mps_overlap(trial, trial)):.10f}")
print(f"E_HF            = {e_hf:.10f}")

# the per-channel energy against the MPO, on non-orthonormal perturbed walkers
_rng = np.random.default_rng(11)
_W = (jnp.asarray(np.stack([Ca + 0.25 * _rng.standard_normal(Ca.shape) for _ in range(32)])),
      jnp.asarray(np.stack([Cb + 0.25 * _rng.standard_normal(Cb.shape) for _ in range(32)])))
_e_fac = np.asarray(jax.jit(jax.vmap(energy_mps))(_W))
_e_mpo = np.asarray(jax.jit(jax.vmap(energy_mps_mpo))(_W))
_e_det = np.asarray(jax.jit(jax.vmap(energy_uhf))(_W))
print(f"\nover 32 perturbed walkers:")
print(f"  max |E_channel - E_MPO|         = {np.abs(_e_fac - _e_mpo).max():.2e}")
print(f"  max |E_channel - E_determinant| = {np.abs(_e_fac - _e_det).max():.2e}")

spin-Schmidt rank of H|psi_T>: 8   <- not 1, so H|psi_T> is NOT a spin
                                    product; the energy below never assumes it is
H|psi_T> before compression: [1, 24, 96, 384, 1536, 384, 96, 24, 1]
H|psi_T> after  compression: [1, 4, 16, 64, 256, 64, 16, 4, 1]

<psi_T|H|psi_T> = -1.5175409663
E_HF            = -1.5175409663

over 32 perturbed walkers:
  max |E_channel - E_MPO|         = 3.11e-15
  max |E_channel - E_determinant| = 1.07e-14


In [31]:
#Running the full MPS AFQMC (est runtime 40/50 s)
def run(overlap_fn, energy_fn=None):
    """Everything is held fixed except the overlap and (optionally) the energy kernel."""
    trial_ops = make_auto_trial_ops(sys_, overlap_u=overlap_fn, get_rdm1=uhf_get_rdm1)
    meas_ops = MeasOps(overlap=overlap_fn,
                       kernels={k_energy: energy_fn if energy_fn is not None else energy_uhf})
    return run_qmc_energy(
        sys=sys_,
        params=params,
        ham_data=ham,
        trial_data=trial_data,
        meas_ops=meas_ops,
        trial_ops=trial_ops,
        prop_ops=prop_ops,
        block_fn=blocks.block,
    )

params = QmcParams(
    dt=0.01,
    n_walkers=100,
    n_prop_steps=10,
    n_blocks=100,
    n_eql_blocks=50,
    weight_floor=1e-8,
    seed=1234,
)

def init_prop_state_pinned(**kwargs):
    """trot's initializer, with a strongly typed node counter.

    trot.prop.cpmc.init_prop_state sets node_encounters = jnp.asarray(0), which is
    WEAKLY typed, while every propagation step hands it back strongly typed
    (int + jnp.sum(bool)). The aval of the state therefore differs between the
    first and the second call of the jitted run_blocks in trot.driver, so the
    whole block scan is traced and compiled TWICE -- which is the two ~19 s
    equilibration blocks at the start of every run. Pinning the dtype keeps the
    aval fixed and costs one compile instead of two. Nothing under trot/ is
    modified.
    """
    state = init_prop_state(**kwargs)
    return state._replace(node_encounters=jnp.zeros((), dtype=int))


prop_ops = dataclasses.replace(
    cpmc_slow.make_prop_ops(ham, sys_.walker_kind),
    init_prop_state=init_prop_state_pinned,
)

mean_mps, err_mps, be_mps, bw_mps = run(overlap_mps, energy_mps)



Equilibration:

        block           E_blk             W        nodes      t[s]
[eql    0/50]   -1.5175409663  1.000000e+02           0       0.0
[eql   10/50]   -3.6930215731  1.144572e+02           0      23.3
[eql   20/50]   -4.2988328956  1.090102e+02           0      32.8
[eql   30/50]   -4.2025070668  1.026028e+02           0      42.3
[eql   40/50]   -4.2334378458  1.010712e+02           0      51.8
[eql   50/50]   -4.3527758910  1.010242e+02           0      61.3

Sampling:

        block           E_avg       E_err         E_block             W         nodes    dt[s/bl]     t[s]
[blk   10/100]   -4.2800242881   2.711e-02     -4.2800242881  9.991677e+01           0      7.083      70.8
[blk   20/100]   -4.2552037623   1.255e-02     -4.2303082243  9.961571e+01           0      0.951      80.3
[blk   30/100]   -4.2259876290   1.421e-02     -4.1673654046  9.944296e+01           0      0.951      89.8
[blk   40/100]   -4.2167602876   1.439e-02     -4.1891845035  1.000424e+02   

In [32]:
#Runining the full SD AFQMC and comparinf to previous run
params = QmcParams(
    dt=0.01,
    n_walkers=100,
    n_prop_steps=10,
    n_blocks=100,
    n_eql_blocks=50,
    weight_floor=1e-8,
    seed=1234,
)
mean_ref, err_ref, be_ref, bw_ref = run(uhf_overlap_u)
be_mps, be_ref = np.asarray(be_mps),np.asarray(be_ref)
print(f"E_HF                            {e_hf:.10f}")
print(f"E_exact (diagonalisation)       {e_exact:.10f}")
print(f"CPMC, all-MPS (overlap+energy)  {float(mean_mps):.10f} +- {float(err_mps):.10f}")
print(f"CPMC, all-determinant           {float(mean_ref):.10f} +- {float(err_ref):.10f}")

d = np.abs(be_mps - be_ref)
print(f"\nmean energy difference:   {abs(float(mean_mps) - float(mean_ref)):.3e}")
print(f"|E_mps - E_det| per block: max {d.max():.3e}   median {np.median(d):.3e}"
      f"   final {d[-1]:.3e}")
print(f"\n Errors per block")
print("  " + "  ".join(f"{x:.0e}" for x in d))



Equilibration:

        block           E_blk             W        nodes      t[s]
[eql    0/50]   -1.5175409663  1.000000e+02           0       0.0
[eql   10/50]   -3.6930215731  1.144572e+02           0       0.6
[eql   20/50]   -4.2988328956  1.090102e+02           0       0.7
[eql   30/50]   -4.2025070668  1.026028e+02           0       0.7
[eql   40/50]   -4.2334378458  1.010712e+02           0       0.8
[eql   50/50]   -4.3527758910  1.010242e+02           0       0.9

Sampling:

        block           E_avg       E_err         E_block             W         nodes    dt[s/bl]     t[s]
[blk   10/100]   -4.2800242881   2.711e-02     -4.2800242881  9.991677e+01           0      0.099       1.0
[blk   20/100]   -4.2552037623   1.255e-02     -4.2303082243  9.961571e+01           0      0.008       1.1
[blk   30/100]   -4.2259876290   1.421e-02     -4.1673654046  9.944296e+01           0      0.008       1.1
[blk   40/100]   -4.2167602876   1.439e-02     -4.1891845035  1.000424e+02   

In [33]:
"""A full MPS CPMC run on the MINIMAL-B plan instead of the maximal one, to see how
the compression error behaves as the walkers propagate. The block sizes are fixed
once at the start and held constant for the whole evolution.

Minimal B DOES need the walker orthonormalised first; maximal B does not care.
With B maximal the selected mode is an exact eigenvector of the whole remaining
block, so the rotation diagonalises Lambda exactly and the orbital space is
preserved for any C. With B minimal the block eigenvector is only an approximate
eigenvector of the full Lambda, and that approximation rests on the spectrum being
{0, 1} -- which is exactly what orthonormality provides. Feed it a raw walker and
the error is ~100%, not ~sqrt(delta). (Both paths now QR the walker and carry
det(R), so the two are on the same footing and this is no longer a difference
between them.)

Minimal B is also why `channel_angles` still has an "eigh" mode. There the
retained block is NOT idempotent, so the cheap exact-projector filter does not
apply, and the repeated-squaring filter needs all 25 squarings to separate the
cluster (n_sq=10 still leaves a 2e-2 error). eigh gets it exactly and is faster
than squaring: 4.7 ms against 5.6 ms per 100 walkers. mode="auto" picks it here.
"""

bp_min_a = bp_min_b = None
if CHI_WALKER is not None or CUTOFF > 0.0:
    bp_min_a = plan_bonds(Ca, plan_a_min, chi_max=CHI_WALKER, cutoff=CUTOFF)
    bp_min_b = plan_bonds(Cb, plan_b_min, chi_max=CHI_WALKER, cutoff=CUTOFF)

trial_min_a, qn_min_a, _sa = channel_mps(jnp.asarray(Ca), plan_a_min, bp_min_a, **_conv_t)
trial_min_b, qn_min_b, _sb = channel_mps(jnp.asarray(Cb), plan_b_min, bp_min_b, **_conv_t)
trial_sign_min = _sa * _sb
trial_min = combine(trial_min_a, qn_min_a, trial_min_b, qn_min_b)[0]   # d=4, unused in the hot path


def overlap_mps_min(walker, trial_data=None):
    """<psi_T|SD(C)> through the minimal-B plan."""
    ca, cb = walker
    qa, ra = _qr(ca)
    qb, rb = _qr(cb)
    ta, _, sa = channel_mps(qa, plan_a_min, bp_min_a, **_conv_w)
    tb, _, sb = channel_mps(qb, plan_b_min, bp_min_b, **_conv_w)
    return (ra * rb * sa * sb * trial_sign_min
            * chan_overlap(trial_min_a, ta) * chan_overlap(trial_min_b, tb))


print("minimal-B channel bonds =",
      [x.shape[0] for x in trial_min_a] + [trial_min_a[-1].shape[-1]])
print("maximal-B channel bonds =",
      [x.shape[0] for x in trial_a] + [trial_a[-1].shape[-1]])

# the two plans and the determinant, on the same batch of perturbed walkers
_rng = np.random.default_rng(11)
_W = (jnp.asarray(np.stack([Ca + 0.25 * _rng.standard_normal(Ca.shape) for _ in range(32)])),
      jnp.asarray(np.stack([Cb + 0.25 * _rng.standard_normal(Cb.shape) for _ in range(32)])))
_ov_min = np.asarray(jax.jit(jax.vmap(overlap_mps_min))(_W))
_ov_max = np.asarray(jax.jit(jax.vmap(overlap_mps))(_W))
_ov_det = np.asarray(jax.vmap(lambda w: uhf_overlap_u(w, trial_data))(_W))
print(f"\noverlap vs determinant over {_W[0].shape[0]} perturbed walkers:")
print(f"  minimal B : max rel {np.abs(_ov_min/_ov_det - 1).max():.2e}")
print(f"  maximal B : max rel {np.abs(_ov_max/_ov_det - 1).max():.2e}")

mean_min, err_min, be_min, bw_min = run(overlap_mps_min)
be_min = np.asarray(be_min)

print(f"\nE_exact (diagonalisation)      {e_exact:.12f}")
print(f"CPMC, minimal-B MPS overlap    {float(mean_min):.12f} +- {float(err_min):.2e}")
print(f"CPMC, determinant overlap      {float(mean_ref):.12f} +- {float(err_ref):.2e}")
print(f"mean-energy difference         {abs(float(mean_min) - float(mean_ref)):.3e}")

d = np.abs(be_min - np.asarray(be_ref))
print(f"\n|E_minB - E_det| per block: max {d.max():.3e}   median {np.median(d):.3e}"
      f" final {d[-1]:.3e}")
print(f"\n Errors per block")
print("  " + "  ".join(f"{x:.0e}" for x in d))

minimal-B channel bonds = [1, 2, 4, 8, 16, 8, 4, 2, 1]
maximal-B channel bonds = [1, 2, 4, 8, 16, 8, 4, 2, 1]

overlap vs determinant over 32 perturbed walkers:
  minimal B : max rel 4.00e-15
  maximal B : max rel 4.77e-15

Equilibration:

        block           E_blk             W        nodes      t[s]
[eql    0/50]   -1.5175409663  1.000000e+02           0       0.0
[eql   10/50]   -3.6930215731  1.144572e+02           0      15.6
[eql   20/50]   -4.2988328956  1.090102e+02           0      25.0
[eql   30/50]   -4.2025070668  1.026028e+02           0      34.4
[eql   40/50]   -4.2334378458  1.010712e+02           0      43.6
[eql   50/50]   -4.3527758910  1.010242e+02           0      52.8

Sampling:

        block           E_avg       E_err         E_block             W         nodes    dt[s/bl]     t[s]
[blk   10/100]   -4.2800242881   2.711e-02     -4.2800242881  9.991677e+01           0      6.191      61.9
[blk   20/100]   -4.2552037623   1.255e-02     -4.2303082243  9.961571

In [34]:
"""THE FAST SWEEP -- the whole discrete Hubbard-Stratonovich site loop for one
walker, off a SINGLE conversion.

The naive CPMC step reconverts the walker to an MPS and recontracts the overlap
twice per site, 2L times per step. This does one conversion and then walks the
site loop with environments, which is where nearly all of the speed comes from.

Three things make it cheap.

1. THE FIELD OPERATOR FACTORISES OVER SPIN. At site x the two discrete choices are
   diagonal one-site operators, in basis l = n_a + 2 n_b

        D_f = [1, h_fa, h_fb, h_fa h_fb] = diag[1, h_fa] (x) diag[1, h_fb]

   so the sweep runs independently in the two channels at chi_sigma, never once at
   chi_a * chi_b. At L=8 that is 16 and 16 instead of 256.

2. ONE O(chi^3) CONTRACTION PER SITE AND CHANNEL, NOT TWO. Both the marginal and
   the pushed-forward left environment need the same intermediate

        T[l,c,d] = Lenv[a,b] bra[a,l,c] ket[b,l,d]

   so it is built once and reused:

        M[l]       = T[l,c,d] R[x+1][c,d]     (O(2 chi^2))
        Lenv'[c,d] = T[0] + h T[1]            (O(2 chi^2))

3. R IS BUILT ONCE AND STAYS VALID. The walker C is updated row by row as fields
   are chosen, but the MPS is never rebuilt: sites already passed are folded into
   Lenv, sites still ahead are untouched. R[0][0,0] also hands back the pre-loop
   overlap for free.

The orthonormalisation det(R) is a single overall factor: row scaling by the field
operator commutes with it, so it rides along in `pref` for the whole sweep rather
than being recomputed.

Measured against the d=4 sweep, 100 walkers: 112.7 ms -> 5.1 ms (22x), with
identical field choices and node counts and overlaps agreeing to 4e-11 -- the
factorised one being the more accurate of the two (4.6e-15 vs 4.0e-11 against the
exact determinant overlap).
"""

def fast_sweep(ca, cb, rns, hs, w_floor):
    """One walker's site loop. Returns (ca, cb, overlap in, overlap out, weight, nodes)."""
    qa, ra = _qr(ca)
    qb, rb = _qr(cb)
    ta, _, sa = channel_mps(qa, plan_a, bp_walk_a, **_conv_w)
    tb, _, sb = channel_mps(qb, plan_b, bp_walk_b, **_conv_w)
    pref = ra * rb * sa * sb * trial_sign

    Ra, Rb = right_envs(trial_a, ta), right_envs(trial_b, tb)
    ov_in = pref * Ra[0][0, 0] * Rb[0][0, 0]
    La = Lb = jnp.ones((1, 1))
    ov, logw = ov_in, jnp.zeros(())
    nodes = jnp.zeros((), jnp.int32)

    for x in range(L):
        Ta = jnp.einsum("ab,alc,bld->lcd", La, trial_a[x], ta[x])   # the only O(chi^3)
        Tb = jnp.einsum("ab,alc,bld->lcd", Lb, trial_b[x], tb[x])
        Ma = jnp.einsum("lcd,cd->l", Ta, Ra[x + 1])
        Mb = jnp.einsum("lcd,cd->l", Tb, Rb[x + 1])

        ov0 = pref * (Ma[0] + hs[0, 0] * Ma[1]) * (Mb[0] + hs[0, 1] * Mb[1])
        ov1 = pref * (Ma[0] + hs[1, 0] * Ma[1]) * (Mb[0] + hs[1, 1] * Mb[1])
        r0 = jnp.where(0.5 * (ov0 / ov) < w_floor, 0.0, 0.5 * (ov0 / ov))
        r1 = jnp.where(0.5 * (ov1 / ov) < w_floor, 0.0, 0.5 * (ov1 / ov))
        nodes = nodes + (r0 <= 0.0) + (r1 <= 0.0)        # the constrained-path test
        norm = r0 + r1 + 1.0e-13
        take0 = rns[x] < r0 / norm
        da = jnp.where(take0, hs[0, 0], hs[1, 0])
        db = jnp.where(take0, hs[0, 1], hs[1, 1])
        ov = jnp.where(take0, ov0, ov1)
        logw = logw + jnp.log(norm)
        ca = ca.at[x, :].mul(da)
        cb = cb.at[x, :].mul(db)
        La = Ta[0] + da * Ta[1]                          # reuses Ta
        Lb = Tb[0] + db * Tb[1]
    return ca, cb, ov_in, ov, jnp.exp(logw), nodes


def make_fast_prop_ops(ham_data, walker_kind):
    """PropOps using the sweep instead of 2L reconversions.

    Same arithmetic as trot.prop.cpmc_slow.cpmc_step -- same RNG stream, same
    weight clamps, same population control -- so the two routes are comparable
    step for step, not merely statistically. Nothing under trot/ is modified.
    """
    cpmc_ops = make_hubbard_cpmc_ops(ham_data, walker_kind)

    def step(state, *, params, ham_data, trial_data, trial_ops,
             meas_ops, meas_ctx, prop_ctx):
        key, subkey = jax.random.split(state.rng_key)
        nw = wk.n_walkers(state.walkers)
        rns = jax.random.uniform(subkey, (nw, cpmc_ops.n_sites()))
        w_floor = float(getattr(params, "weight_floor", 1.0e-8))
        w_cap = float(getattr(params, "weight_cap", 100.0))
        damping = float(getattr(params, "pop_control_damping", 0.1))

        # --- first one-body half step, then the whole site loop on one conversion ---
        walkers = cpmc_ops.apply_one_body_half(state.walkers, prop_ctx)
        ca, cb, ov_half, overlaps, wfac, nod = jax.vmap(
            fast_sweep, in_axes=(0, 0, 0, None, None))(
            walkers[0], walkers[1], rns, prop_ctx.hs_constant, w_floor)

        ratio = jnp.real(jnp.real(ov_half) / state.overlaps)
        ratio = jnp.where(ratio < w_floor, 0.0, ratio)
        nodes = jnp.sum(ratio <= 0.0) + jnp.sum(nod)
        weights = jnp.where(state.weights * ratio > w_cap, 0.0, state.weights * ratio)
        weights = weights * wfac
        walkers = (ca, cb)

        # --- second one-body half step: a general rotation, so a full overlap ---
        walkers = cpmc_ops.apply_one_body_half(walkers, prop_ctx)
        overlaps_new = jnp.real(
            jax.vmap(meas_ops.overlap, in_axes=(0, None))(walkers, trial_data))
        ratio = jnp.real(overlaps_new / overlaps)
        ratio = jnp.where(ratio < w_floor, 0.0, ratio)
        nodes = nodes + jnp.sum(ratio <= 0.0)
        weights = jnp.where(weights * ratio > w_cap, 0.0, weights * ratio)

        weights = weights * jnp.exp(prop_ctx.dt * state.pop_control_ene_shift)
        weights = jnp.where(weights > w_cap, 0.0, weights)
        avg_w = jnp.clip(jnp.mean(weights), min=1.0e-300)
        return PropState(
            walkers=walkers, weights=weights, overlaps=overlaps_new, rng_key=key,
            pop_control_ene_shift=state.e_estimate
            - damping * (jnp.log(avg_w) / prop_ctx.dt),
            e_estimate=state.e_estimate,
            node_encounters=state.node_encounters + nodes,
        )

    return PropOps(init_prop_state=init_prop_state_pinned,
                   build_prop_ctx=lambda h, t, p: _build_prop_ctx(h, p.dt),
                   step=step)


prop_ops_slow = prop_ops
prop_ops_fast = make_fast_prop_ops(ham, sys_.walker_kind)

prop_ops = prop_ops_fast
mean_fast, err_fast, be_fast, bw_fast = run(overlap_mps, energy_mps)
prop_ops = prop_ops_slow

be_fast = np.asarray(be_fast)

print(f"E_exact (diagonalisation)        {e_exact:.12f}")
print(f"CPMC, MPS + fast sweep           {float(mean_fast):.12f} +- {float(err_fast):.2e}")
print(f"CPMC, MPS + reconversion         {float(mean_mps):.12f} +- {float(err_mps):.2e}")
print(f"CPMC, all-determinant            {float(mean_ref):.12f} +- {float(err_ref):.2e}")

print(f"\nmean difference, fast vs reconversion  {abs(float(mean_fast) - float(mean_mps)):.3e}")
print(f"mean difference, fast vs determinant   {abs(float(mean_fast) - float(mean_ref)):.3e}")

d = np.abs(be_fast - be_mps)
print(f"\n|E_fast - E_reconv| per block: max {d.max():.3e}   median {np.median(d):.3e}")
print("  " + "  ".join(f"{x:.0e}" for x in d))


Equilibration:

        block           E_blk             W        nodes      t[s]
[eql    0/50]   -1.5175409663  1.000000e+02           0       0.0
[eql   10/50]   -3.6930215731  1.144572e+02           0      10.5
[eql   20/50]   -4.2988328956  1.090102e+02           0      11.7
[eql   30/50]   -4.2025070668  1.026028e+02           0      12.8
[eql   40/50]   -4.2334378458  1.010712e+02           0      14.0
[eql   50/50]   -4.3527758910  1.010242e+02           0      15.1

Sampling:

        block           E_avg       E_err         E_block             W         nodes    dt[s/bl]     t[s]
[blk   10/100]   -4.2800242881   2.711e-02     -4.2800242881  9.991677e+01           0      1.634      16.3
[blk   20/100]   -4.2552037623   1.255e-02     -4.2303082243  9.961571e+01           0      0.115      17.5
[blk   30/100]   -4.2259876290   1.421e-02     -4.1673654046  9.944296e+01           0      0.114      18.6
[blk   40/100]   -4.2167602876   1.439e-02     -4.1891845035  1.000424e+02   

In [35]:
#Comparing perfomances of pure jax overlap with pyblock3
import time

import pyblock3.algebra.ad as ad

ad.ENABLE_JAX = True                    # must precede the ad submodule imports
from pyblock3.algebra.symmetry import SZ
from pyblock3.algebra.ad.core import SparseTensor, SubTensor
from pyblock3.algebra.ad.mps import MPS

PHYS = np.array([[li % 2, li // 2] for li in range(4)])     # local index -> (n_a, n_b)
Q = lambda na, nb: SZ(int(na) + int(nb), int(na) - int(nb), 0)


def to_pyblock3(tensors, qn):
    """Dense (Dl, 4, Dr) arrays + (n_a, n_b) bond labels -> block-sparse pyblock3 MPS.

    A block exists only where left label + physical label = right label, which is
    what makes the representation sparse.
    """
    out = []
    for i, A in enumerate(tensors):
        blocks = []
        for nl in sorted(set(map(tuple, qn[i].tolist()))):
            rows = np.where((qn[i] == np.array(nl)).all(1))[0]
            for li in range(4):
                nr = tuple(np.array(nl) + PHYS[li])
                cols = np.where((qn[i + 1] == np.array(nr)).all(1))[0]
                if len(cols) == 0:
                    continue
                data = A[np.ix_(rows, [li], cols)]
                if float(jnp.abs(data).max()) == 0.0:
                    continue
                blocks.append(SubTensor(data=data,
                                        q_labels=(Q(*nl), Q(*PHYS[li]), Q(*nr))))
        out.append(SparseTensor(blocks=blocks))
    return MPS(tensors=out)


# combine() returns the bond labels too; sd_to_mps drops them, so call it directly
_ta, _qna, _ = channel_mps(jnp.asarray(Ca), plan_a)
_tb, _qnb, _ = channel_mps(jnp.asarray(Cb), plan_b)
ket_ts, ket_qn = combine(_ta, _qna, _tb, _qnb)
ket_pb = to_pyblock3(ket_ts, ket_qn)

# a batch of perturbed, row-scaled, non-orthonormal walkers -- built here so this
# appendix stands on its own
_rng = np.random.default_rng(0)
_nw = 32
_wu = np.stack([Ca + 0.25 * _rng.standard_normal(Ca.shape) for _ in range(_nw)])
_wd = np.stack([Cb + 0.25 * _rng.standard_normal(Cb.shape) for _ in range(_nw)])
_wu[:, 3, :] *= 2.1
_wd[:, 5, :] *= 0.4
Wb = (jnp.asarray(_wu), jnp.asarray(_wd))

qa0, _ = jnp.linalg.qr(Wb[0][0])
qb0, _ = jnp.linalg.qr(Wb[1][0])
_ta, _qna, _ = channel_mps(qa0, plan_a)
_tb, _qnb, _ = channel_mps(qb0, plan_b)
bra_ts, bra_qn = combine(_ta, _qna, _tb, _qnb)
bra_pb = to_pyblock3(bra_ts, bra_qn)

ours_val = float(mps_overlap(bra_ts, ket_ts))
pb3_val = float(bra_pb.dot(ket_pb))
print(f"blocks are jax arrays : {isinstance(ket_pb.tensors[1].blocks[0].data, jnp.ndarray)}")
print(f"<bra|ket>  dense      : {ours_val:.14f}")
print(f"<bra|ket>  pyblock3   : {pb3_val:.14f}")
print(f"           rel diff   : {abs(pb3_val / ours_val - 1):.3e}")

# ---------------------------------------------------------------- how sparse is it?
dense_n = sum(int(np.prod(t.shape)) for t in bra_ts)
block_n = sum(int(np.prod(b.data.shape)) for t in bra_pb.tensors for b in t.blocks)
print(f"\nblocks per site : {[len(t.blocks) for t in bra_pb.tensors]}")
print(f"dense bond dims : {[t.shape[0] for t in bra_ts] + [bra_ts[-1].shape[-1]]}")
print(f"stored entries  : {block_n} of {dense_n} dense  ->  {100*block_n/dense_n:.1f}%")


# ------------------------------------------------- contraction cost, same inputs
def bench(f, n=20):
    f()
    t0 = time.time()
    for _ in range(n):
        f()
    return (time.time() - t0) / n * 1e3


ket_np, bra_np = [np.asarray(t) for t in ket_ts], [np.asarray(t) for t in bra_ts]


def mps_overlap_np(bra, ket):
    e = np.ones((bra[0].shape[0], ket[0].shape[0]))
    for a, b in zip(bra, ket):
        e = np.tensordot(a, np.tensordot(e, b, ([1], [0])), ([0, 1], [0, 1]))
    return float(e.reshape(()))


jit_ov = jax.jit(mps_overlap)
jit_ov(bra_ts, ket_ts).block_until_ready()

print("\ncontraction only, tensors already built, per walker:")
print(f"  dense tensordot, numpy    : {bench(lambda: mps_overlap_np(bra_np, ket_np)):8.3f} ms")
print(f"  dense tensordot, jnp      : {bench(lambda: float(mps_overlap(bra_ts, ket_ts))):8.3f} ms")
print(f"  dense tensordot, jitted   : {bench(lambda: jit_ov(bra_ts, ket_ts).block_until_ready()):8.3f} ms")
print(f"  pyblock3 .dot             : {bench(lambda: float(bra_pb.dot(ket_pb))):8.3f} ms")
print(f"\nand just building the pyblock3 MPS : {bench(lambda: to_pyblock3(bra_ts, bra_qn)):8.3f} ms")

# ------------------------------------------------- end to end over the batch
f_ours = jax.jit(jax.vmap(lambda w: mps_overlap(
    sd_to_mps(jnp.linalg.qr(w[0])[0], jnp.linalg.qr(w[1])[0], plan_a, plan_b), ket_ts)))
f_ours(Wb).block_until_ready()
t_ours = bench(lambda: f_ours(Wb).block_until_ready(), n=10)


def pb3_batch():
    out = []
    for k in range(Wb[0].shape[0]):
        qa, _ = jnp.linalg.qr(Wb[0][k])
        qb, _ = jnp.linalg.qr(Wb[1][k])
        _a, _qa, _ = channel_mps(qa, plan_a)
        _b, _qb, _ = channel_mps(qb, plan_b)
        ts, qn = combine(_a, _qa, _b, _qb)
        out.append(float(to_pyblock3(ts, qn).dot(ket_pb)))
    return np.array(out)


r_pb3 = pb3_batch()
t_pb3 = bench(pb3_batch, n=1)
nw = Wb[0].shape[0]
print(f"\nmax rel diff over {nw} walkers : {np.max(np.abs(r_pb3 / np.asarray(f_ours(Wb)) - 1)):.3e}")
print(f"end to end, {nw} walkers:")
print(f"  ours, jit + vmap          : {t_ours:9.2f} ms  ({t_ours/nw:7.3f} ms/walker)")
print(f"  pyblock3, python loop     : {t_pb3:9.2f} ms  ({t_pb3/nw:7.2f} ms/walker)"
      f"   -> {t_pb3/t_ours:.0f}x slower")

blocks are jax arrays : True
<bra|ket>  dense      : -0.35806205341943
<bra|ket>  pyblock3   : -0.35806205341943
           rel diff   : 4.441e-16

blocks per site : [4, 16, 36, 64, 64, 36, 16, 4]
dense bond dims : [1, 4, 16, 64, 256, 64, 16, 4, 1]
stored entries  : 10680 of 139808 dense  ->  7.6%

contraction only, tensors already built, per walker:
  dense tensordot, numpy    :    0.333 ms
  dense tensordot, jnp      :    1.435 ms
  dense tensordot, jitted   :    1.080 ms
  pyblock3 .dot             :   64.352 ms

and just building the pyblock3 MPS :   89.472 ms

max rel diff over 32 walkers : 2.331e-15
end to end, 32 walkers:
  ours, jit + vmap          :     12.64 ms  (  0.395 ms/walker)
  pyblock3, python loop     :   8855.95 ms  ( 276.75 ms/walker)   -> 701x slower


In [36]:
"""PERFECT SAMPLING FROM AN MPS.

Draws whole occupation configurations n = (n_0 ... n_{L-1}) with probability
EXACTLY

    p(n) = |<n|psi>|^2 / <psi|psi>

in one left-to-right pass: no Markov chain, no burn-in, no autocorrelation, no
rejections. Ferris & Vidal, PRB 85, 165146 (2012); the writeup at
tensornetwork.org/mps/algorithms/sampling; the algorithm ITensorMPS implements
as `sample!`.

HOW IT WORKS. Factor p(n) = p(n_0) p(n_1|n_0) ... and sample the conditionals in
order. Each conditional is a one-site reduced density matrix with the sites to
the LEFT projected onto what has already been drawn and the sites to the RIGHT
traced out. Tracing out the right half is free if the MPS is right-canonical,
sum_s B_x[:,s,:] B_x[:,s,:]^+ = I, because then everything from x+1 on contracts
to the identity. So with a NORMALISED RIGHT-CANONICAL MPS and a running left
vector v (||v|| = 1 throughout),

    w[s, :] = v . A_x[:, s, :]        p(s | n_<x) = ||w[s]||^2

and sum_s ||w[s]||^2 = v (sum_s A A^+) v^+ = ||v||^2 = 1 exactly -- the
conditional comes out normalised, there is nothing to divide by and no
denominator that can vanish. Draw s, set v <- w[s]/||w[s]||, step right. One
sample costs O(L d chi^2): the same as ONE overlap, and a factor chi less than
the O(L d chi^3) of contracting two MPS together.

WHY IT BELONGS IN THIS NOTEBOOK. Everything else here is an exact contraction,
which is the right thing to do at chi = 16. Sampling is what takes over when it
is not: for ANY operator

    <psi|O|psi> / <psi|psi> = E_{n ~ p} [ <n|O|psi> / <n|psi> ]

and both amplitudes are O(L d chi^2) row-vector sweeps, so the estimator never
contracts two MPS with each other at all. Measured on random right-canonical
MPS, L = 40, one CPU core, float64:

    chi    1 sample   1k batched   /sample   one <a|b>   samples per overlap
     32     0.13 ms       4.0 ms    4.0 us     0.25 ms            63
    128     0.24 ms      11.9 ms   11.9 us     7.86 ms           660
    512     3.22 ms      66.4 ms   66.4 us    60.73 ms           914

It is also how you draw configurations FROM a trial MPS, which is what an
initialisation or a reweighting step needs.

CONVENTIONS -- the same as the rest of the notebook. An MPS is a python list of
L arrays of shape (Dl, d, Dr) with D_0 = D_L = 1. d = 2 for one spin channel
(s = n_x), d = 4 for the combined state (s = n_alpha + 2 n_beta). L is a python
int, so the loops unroll at trace time and everything here is jit- and
vmap-able with static shapes. Complex tensors work (conjugations are in place
and are free on real input), though this notebook is real throughout.

PARTICLE NUMBER IS AUTOMATIC. `channel_mps` produces an MPS with exact U(1)
quantum numbers, so configurations with the wrong electron count have amplitude
identically zero and are drawn with probability zero. No projection, no
rejection -- the next cell checks that not one sample in 200000 comes out with
the wrong filling.
"""
import time

Sample = namedtuple("Sample", "config logp amp")


# ---------------------------------------------------------------- canonical form
def right_canonicalize(ts, normalize=True):
    """Right-to-left sweep to right-canonical form, orthogonality centre on site 0.

    sum_s B_x[:, s, :] B_x[:, s, :]^+ = I for every x >= 1 -- the property that
    makes the sampling conditionals normalised by construction.

    Done as an LQ, i.e. a QR of the conjugate transpose: with
    M = A_x.reshape(Dl, d*Dr) and M^+ = Q R, the right factor is Q^+ (orthonormal
    ROWS, so right-isometric) and R^+ is pushed one site left. Bond dimensions
    can only shrink, to min(Dl, d*Dr), and they are python ints, so shapes stay
    static and this composes with vmap.

    Returns (tensors, norm). `normalize` divides site 0 by the norm, which is the
    state the sampler wants; the norm is handed back because an amplitude of the
    ORIGINAL state is norm * <n|psi_normalised>.

    Run this ONCE per MPS, then draw as many samples as you like -- it is off the
    hot path. The MPS out of `channel_mps` is NOT right-canonical (its splits put
    the orthogonality centre on the right), so it does need the sweep.
    """
    ts = [jnp.asarray(t) for t in ts]
    for x in range(len(ts) - 1, 0, -1):
        Dl, d, Dr = ts[x].shape
        q, r = jnp.linalg.qr(ts[x].reshape(Dl, d * Dr).conj().T, mode="reduced")
        ts[x] = q.conj().T.reshape(-1, d, Dr)
        ts[x - 1] = jnp.tensordot(ts[x - 1], r.conj().T, axes=([2], [0]))
    norm = jnp.linalg.norm(ts[0])
    if normalize:
        ts[0] = ts[0] / jnp.where(norm == 0.0, 1.0, norm)
    return ts, norm


def canonical_error(ts):
    """max_x || sum_s B_x B_x^+ - I ||_inf over x >= 1. Zero iff right-canonical."""
    return max(float(jnp.abs(jnp.einsum("axc,bxc->ab", t, t.conj())
                             - jnp.eye(t.shape[0])).max()) for t in ts[1:])


# --------------------------------------------------------------------- the draw
def perfect_sample(ts, key=None, us=None, eps=1e-300):
    """One perfect sample from a NORMALISED RIGHT-CANONICAL MPS.

    Returns (config, logp, amp):
      config  (L,) int32 local indices; n_x for d=2, n_a + 2 n_b for d=4
      logp    log p(config), the exact log probability this draw was made with
      amp     <config|psi> of the NORMALISED state, sign (or phase) included

    |amp|^2 == exp(logp) up to roundoff. Both are returned because ratio
    estimators need the signed amplitude while reweighting needs the probability.

    Randomness enters only as L uniforms, either drawn from `key` or supplied
    directly as `us` -- the same pattern as `fast_sweep`, so a sampled step can
    be driven off the same RNG stream as a propagation step.

    TWO THINGS ARE DELIBERATE ABOUT THE INNER LOOP, both worth 1.06-1.6x and
    neither changing a single draw (checked bit for bit):

      * the site index is an UNROLLED inverse CDF, d-1 comparisons summed,
        rather than `jnp.searchsorted` (which is a loop XLA cannot flatten at
        d = 2 or 4) or `jax.random.categorical` (which would take the log of a
        probability that is legitimately zero). Measured on random
        right-canonical MPS, batched over samples:

            L, chi        searchsorted   unrolled
             8,   16        0.18 us        0.11 us
            32,   64        3.79 us        2.88 us
            32,  256       27.99 us       24.70 us
            64,  512      251.48 us      236.14 us

        The gain shrinks with chi because the GEMM takes over, which is the
        point: past chi ~ 256 this loop is already at the hardware limit. On the
        trial of this notebook the self-check cell measures 0.14 -> 0.09 us.

      * `logp` is NOT accumulated. The state is normalised and right-canonical,
        so the conditionals sum to 1 exactly and log p = 2 log|amp| identically;
        carrying both costs L logarithms to reproduce a number already in hand.
    """
    n = len(ts)
    us = jax.random.uniform(key, (n,)) if us is None else us
    v = jnp.ones((1,), ts[0].dtype)              # left vector, ||v|| = 1 throughout
    cfg, logamp = [], jnp.zeros(())
    for x in range(n):
        w = jnp.tensordot(v, ts[x], axes=([0], [0]))       # (d, Dr)
        p = jnp.einsum("sr,sr->s", w, w.conj()).real       # sums to ||v||^2 = 1
        c = jnp.cumsum(p)
        u = us[x] * c[-1]                                  # c[-1] = 1 up to roundoff
        s = sum((u >= c[k]).astype(jnp.int32) for k in range(p.shape[0] - 1))
        nrm = jnp.sqrt(jnp.maximum(p[s], eps))             # guard: p[s] > 0 a.s.
        v = w[s] / nrm
        cfg.append(s)
        logamp = logamp + jnp.log(nrm)
    amp = jnp.exp(logamp) * v[0]
    return Sample(jnp.stack(cfg).astype(jnp.int32), 2.0 * logamp, amp)


def perfect_sample_batch(ts, key, n_samples):
    """`n_samples` independent perfect samples, vmapped over split keys."""
    return jax.vmap(perfect_sample, in_axes=(None, 0))(ts, jax.random.split(key, n_samples))


def sample_mps(ts, key, n_samples=None):
    """Canonicalise, normalise, then draw. The convenience entry point.

    This is ITensorMPS' `sample!` (orthogonalise first); `perfect_sample` is its
    `sample` (centre already on site 1). Returns (samples, norm) -- multiply
    `amp` by `norm` for amplitudes of the state as it was handed in.
    """
    ts, norm = right_canonicalize(ts)
    s = perfect_sample(ts, key) if n_samples is None else perfect_sample_batch(ts, key, n_samples)
    return s, norm


# ----------------------------------------------------------------- amplitudes
def mps_amplitude(ts, config):
    """<config|psi>, O(L d chi^2). No canonical form needed.

    Contracts ALL d local states and then picks, instead of the obvious
    `v @ A[:, config[x], :]`. That does d times the arithmetic and is far
    faster, because it is the difference between one shared GEMM and a
    per-sample gather of a whole chi x chi matrix: under vmap the obvious form
    materialises an (N, chi, chi) tensor at every site and goes memory bound.
    Measured, L = 16, batched over N configurations, identical results to 3e-18:

        chi       N     gather      contract-then-pick    speedup
         32   20000     49.3 ms           6.5 ms            7.6x
         64   20000    542.9 ms          22.3 ms           24.4x
        128   10000    942.1 ms          24.9 ms           37.8x
        256    4000    751.6 ms          32.0 ms           23.5x

    It also compiles ~6x faster; the gather form's compile time grows like chi^2
    and becomes the dominant cost well before chi = 512.
    """
    v = jnp.ones((1,), ts[0].dtype)
    for x, A in enumerate(ts):
        v = jnp.tensordot(v, A, axes=([0], [0]))[config[x]]
    return v[0]


def mps_amplitudes(ts, configs):
    """`mps_amplitude` over a batch of configurations."""
    return jax.vmap(mps_amplitude, in_axes=(None, 0))(ts, configs)


# -------------------------------------------------- the spin-product shortcut
def jw_sign(na, nb):
    """(-1)^{sum_i na_i sum_{j<i} nb_j} -- the interleaving sign `combine` inserts.

    <n_a, n_b| psi_a (x) psi_b> = jw_sign(n_a, n_b) <n_a|psi_a> <n_b|psi_b>, the
    d=4 configuration being na + 2 nb site by site. Probabilities never see it
    (it squares to 1); amplitudes do.
    """
    return (-1.0) ** jnp.sum(na * (jnp.cumsum(nb) - nb))


def sample_spin_product(ta, tb, key, n_samples=None):
    """Sample a SPIN-PRODUCT state by sampling its two channels independently.

    p(n_a, n_b) = p_a(n_a) p_b(n_b) exactly when the state is a spin product, so
    this runs at chi_sigma twice instead of once at chi_a * chi_b -- the same
    factorisation the overlap, the energy and the fast sweep use, with the same
    validity condition, `spin_schmidt_rank(...) == 1`. Walkers and a
    single-determinant trial qualify; a DMRG or multi-determinant trial does NOT,
    and there you sample the combined MPS with `sample_mps`.

    Returns ((sa, sb), norm_a * norm_b). The d=4 configuration is
    sa.config + 2 * sb.config, its amplitude jw_sign * sa.amp * sb.amp * norm.
    """
    ka, kb = jax.random.split(key)
    sa, na_ = sample_mps(ta, ka, n_samples)
    sb, nb_ = sample_mps(tb, kb, n_samples)
    return (sa, sb), na_ * nb_


# ------------------------------------------------------------- the estimator
def ratio_estimator(psi, o_psi, key, n_samples):
    """<psi|O|psi> / <psi|psi> by perfect sampling, with its Monte Carlo error.

    `o_psi` is O|psi> as an MPS -- here `compress(apply_mpo(hubbard_mpo(...), psi))`.
    Then

        <psi|O|psi>/<psi|psi> = sum_n p(n) <n|O|psi>/<n|psi> = E_{n~p}[e_loc(n)]

    the ordinary variational-Monte-Carlo local estimator, unbiased at ANY number
    of samples (the next cell checks that at 0.5 sigma over 24 independent runs).
    Note that only the sampled state has to be canonicalised: O|psi> is only ever
    evaluated amplitude by amplitude.

    Returns (mean, standard error, the local values). The draws are independent,
    so the error is a plain sqrt(var/n) with no binning -- which is exactly what
    perfect sampling buys over a Metropolis walk.
    """
    cts, norm = right_canonicalize(psi)
    s = perfect_sample_batch(cts, key, n_samples)
    e_loc = mps_amplitudes(o_psi, s.config) / (norm * s.amp)
    return e_loc.mean(), e_loc.std() / np.sqrt(n_samples), e_loc


def overlap_estimator(bra, ket, key, n_samples):
    """<bra|ket>/<ket|ket> by sampling `ket`. The two-MPS contraction, stochastically.

    <bra|ket>/<ket|ket> = sum_n p_ket(n) <bra|n>/<n|ket> = E_{n ~ p_ket}[R(n)],
    one O(L d chi^2) amplitude sweep per sample -- the denominator is free,
    because <n|ket> is what the sampler already returns.

    ITS VARIANCE IS KNOWN IN CLOSED FORM, and it is the whole story (see the
    study cell). With both states normalised, E[R] = S and E[R^2] =
    <bra|bra>/<ket|ket> = 1, so

        Var(R) = 1 - S^2        exactly, for ANY two MPS

    -- independent of chi, of L, of d and of the structure of either state.
    Nothing about the states enters except their overlap.

    Returns (mean, standard error, the sampled ratios).
    """
    s = perfect_sample_batch(ket, key, n_samples)
    r = mps_amplitudes(bra, s.config) / s.amp
    return r.mean(), r.std() / np.sqrt(n_samples), r


def overlap_estimator_mis(bra, ket, key, n_samples):
    """The same overlap, sampling the MIXTURE of both states. Strictly better, same cost.

    The plain estimator draws only from `ket` and averages R = b/k, which is
    unbounded: a configuration that `bra` likes and `ket` does not blows it up.
    Draw instead from q = (b^2 + k^2)/2 -- half the samples from each state --
    and reweight with the balance heuristic (Veach & Guibas):

        g(n) = f/q = 2 b_n k_n / (b_n^2 + k_n^2),   E[g] = S,   |g| <= 1

        Var(g) = sum_n 2 b_n^2 k_n^2/(b_n^2 + k_n^2) - S^2
               <= sum_n |b_n k_n| - S^2  <=  1 - S^2 = Var(R)

    so it is never worse, and it is BOUNDED -- no heavy tail, which also means
    its sample variance can be trusted, while the plain estimator's cannot
    (the study cell shows sigma_hat undershooting the exact 1 - S^2 by 25% at
    small S, because 40000 draws never reach the tail that carries the variance).

    It costs exactly the same. Both estimators need one sampling pass and one
    amplitude sweep per sample: the plain one gets k_n free from the sampler and
    computes b_n; this one gets whichever state it drew from free and computes
    the other. Measured variance reduction on an L=12 chain, exact enumeration
    confirming the formulas:

        S         Var plain   Var MIS   gain
        0.9914     0.017125   0.001195  14.3x
        0.9632     0.072274   0.011818   6.1x
        0.7399     0.452569   0.152177   3.0x
        0.0778     0.993941   0.392156   2.5x

    The gain is largest exactly where it matters -- close to S = 1 is the only
    place sampling a two-MPS contraction can beat contracting it.
    """
    h = n_samples // 2
    kb, kk = jax.random.split(key)
    sb = perfect_sample_batch(bra, kb, h)                   # amp = b_n
    sk = perfect_sample_batch(ket, kk, h)                   # amp = k_n
    b = jnp.concatenate([sb.amp, mps_amplitudes(bra, sk.config)])
    k = jnp.concatenate([mps_amplitudes(ket, sb.config), sk.amp])
    gval = 2.0 * b * k / (b * b + k * k)
    return gval.mean(), gval.std() / np.sqrt(2 * h), gval


# ------------------------------------------------------------------ diagnostics
def exact_probs(ts):
    """Every |<n|psi>|^2/<psi|psi> by dense contraction. Small L only: d^L of them."""
    v = np.asarray(ts[0])[0]
    for A in ts[1:]:
        v = np.tensordot(v, np.asarray(A), axes=([-1], [0]))
    p = np.abs(v.ravel()) ** 2
    return p / p.sum()


def config_index(configs, d):
    """Configurations -> flat row-major index, the ordering `exact_probs` returns."""
    configs = np.asarray(configs)
    return configs @ (d ** np.arange(configs.shape[-1] - 1, -1, -1))

In [37]:
"""Self-checks for the sampler. Est runtime 30 s.

Four things are being checked, in increasing order of how much they would hurt
if they were wrong.

1. CANONICALISATION IS A GAUGE TRANSFORMATION. The right-canonical form has to
   be the same state, amplitude by amplitude, up to the norm it factors out.

2. THE DRAWS HAVE THE RIGHT DISTRIBUTION. Not "looks about right": the exact
   p(n) is available at L=8 by dense contraction, so the empirical frequencies
   get a chi^2 against it, and the total variation distance is compared with
   what pure sampling noise predicts,  E|p_hat - p|/2 ~ sqrt(1/2 pi N) sum_n
   sqrt(p_n (1 - p_n)). A biased sampler shows up here as a TV that does not
   shrink as 1/sqrt(N).

3. THE QUANTUM NUMBERS SURVIVE. Every draw must have exactly n_up alpha and
   n_down beta electrons; that is a property of the MPS, not of the sampler, but
   it is the cheapest possible check that the two agree.

4. THE ESTIMATOR IS UNBIASED. <psi_T|H|psi_T>/<psi_T|psi_T> = E_HF is known
   exactly here, so 24 independent runs must straddle it, and their scatter must
   match the error bar the estimator reports for itself.
"""
KEY = jax.random.PRNGKey(0)
N   = 200_000

# ------------------------------------------------- 1. canonicalisation is a gauge
can_a, norm_a = right_canonicalize(trial_a)
can_4, norm_4 = right_canonicalize(trial)
_cfg2 = jnp.asarray([[(i >> k) & 1 for k in range(L - 1, -1, -1)] for i in range(2 ** L)])
_amp_raw = np.asarray(mps_amplitudes(trial_a, _cfg2))
_amp_can = np.asarray(mps_amplitudes(can_a, _cfg2)) * float(norm_a)
print(f"canonical error, channel / d=4   {canonical_error(can_a):.1e} / {canonical_error(can_4):.1e}")
print(f"norm                             {float(norm_a):.12f}")
print(f"max |amp - norm * amp_canonical| {np.abs(_amp_raw - _amp_can).max():.1e}")

# -------------------------------------------------------- 2/3. the distribution
p_exact = exact_probs(can_a)
draw = jax.jit(lambda k: perfect_sample_batch(can_a, k, N))
_t0 = time.perf_counter(); s = draw(KEY); jax.block_until_ready(s)
_t1 = time.perf_counter(); s = draw(jax.random.PRNGKey(1)); jax.block_until_ready(s)
_t2 = time.perf_counter()
cfg = np.asarray(s.config)
idx = config_index(cfg, 2)
hist = np.bincount(idx, minlength=2 ** L) / N
supp = p_exact > 1e-12
tv_pred = 0.5 * np.sqrt(2 / np.pi / N) * np.sqrt(p_exact * (1 - p_exact)).sum()
chi2 = (((hist - p_exact)[supp] * N) ** 2 / (N * p_exact[supp])).sum()
print(f"\none channel, {N} samples: {1e3*(_t2-_t1):.0f} ms  "
      f"({1e6*(_t2-_t1)/N:.2f} us/sample, {1e3*(_t1-_t0):.0f} ms to compile)")
print(f"  total variation |p_hat - p|/2   {0.5*np.abs(hist-p_exact).sum():.4f}"
      f"   (sampling noise predicts {tv_pred:.4f})")
print(f"  chi^2                           {chi2:.1f} on {int(supp.sum())-1} dof")
print(f"  draws outside the support       {int((~supp[idx]).sum())}")
print(f"  draws with n_alpha != {n_up}         {int((cfg.sum(1) != n_up).sum())}")
print(f"  max | |amp|^2 - exp(logp) |     {np.abs(np.asarray(s.amp)**2 - np.exp(np.asarray(s.logp))).max():.1e}")
print(f"  max |amp - <n|psi>/norm|        {np.abs(np.asarray(s.amp) - _amp_raw[idx]/float(norm_a)).max():.1e}")

# ----------------------------------- the combined state, both ways of drawing it
M = 100_000
p4 = exact_probs(can_4)
s4, _ = sample_mps(trial, jax.random.PRNGKey(2), M)
h4 = np.bincount(config_index(np.asarray(s4.config), 4), minlength=4 ** L) / M
(sa, sb), nrm_ab = sample_spin_product(trial_a, trial_b, jax.random.PRNGKey(3), M)
cfg_ab = np.asarray(sa.config) + 2 * np.asarray(sb.config)
h_ab = np.bincount(config_index(cfg_ab, 4), minlength=4 ** L) / M
tv4_pred = 0.5 * np.sqrt(2 / np.pi / M) * np.sqrt(p4 * (1 - p4)).sum()
amp_fac = (np.asarray(jax.vmap(jw_sign)(1.0 * np.asarray(sa.config), 1.0 * np.asarray(sb.config)))
           * np.asarray(sa.amp) * np.asarray(sb.amp) * float(nrm_ab))
print(f"\nd=4, {M} samples over {int((p4>1e-12).sum())} supported configurations "
      f"(sampling noise predicts TV {tv4_pred:.4f}):")
print(f"  TV, sampling the combined MPS   {0.5*np.abs(h4-p4).sum():.4f}")
print(f"  TV, sampling the two channels   {0.5*np.abs(h_ab-p4).sum():.4f}")
print(f"  max |factorised amp - d=4 amp|  "
      f"{np.abs(amp_fac - np.asarray(mps_amplitudes(trial, jnp.asarray(cfg_ab)))).max():.1e}")

# ------------------------------------------------------- 4. the estimator is unbiased
_m = np.array([float(ratio_estimator(trial, Hket, jax.random.PRNGKey(100 + i), 20_000)[0])
               for i in range(24)])
_e = float(ratio_estimator(trial, Hket, KEY, 20_000)[1])
_sem = _m.std(ddof=1) / np.sqrt(len(_m))
print(f"\n<psi_T|H|psi_T>/<psi_T|psi_T> by sampling, 24 independent runs of 20000:")
print(f"  mean of the runs   {_m.mean():.8f}   exact (= E_HF) {e_hf:.8f}")
print(f"  off by             {abs(_m.mean()-e_hf):.1e} = {abs(_m.mean()-e_hf)/_sem:.2f} sigma")
print(f"  scatter of runs    {_m.std(ddof=1):.1e}   vs the error each run reports, {_e:.1e}")

canonical error, channel / d=4   5.6e-16 / 1.3e-15
norm                             1.000000000000
max |amp - norm * amp_canonical| 1.7e-16

one channel, 200000 samples: 18 ms  (0.09 us/sample, 344 ms to compile)
  total variation |p_hat - p|/2   0.0062   (sampling noise predicts 0.0061)
  chi^2                           69.6 on 69 dof
  draws outside the support       0
  draws with n_alpha != 4         0
  max | |amp|^2 - exp(logp) |     1.4e-17
  max |amp - <n|psi>/norm|        1.9e-16

d=4, 100000 samples over 4900 supported configurations (sampling noise predicts TV 0.0605):
  TV, sampling the combined MPS   0.0595
  TV, sampling the two channels   0.0601
  max |factorised amp - d=4 amp|  1.1e-16

<psi_T|H|psi_T>/<psi_T|psi_T> by sampling, 24 independent runs of 20000:
  mean of the runs   -1.51490763   exact (= E_HF) -1.51754097
  off by             2.6e-03 = 0.54 sigma
  scatter of runs    2.4e-02   vs the error each run reports, 2.5e-02


In [39]:
"""CPMC ON A TRIAL BUILT BY SAMPLING THE MPS.  Est runtime 90 s.

A deliberately roundabout route, as a test of the sampler: take the HF
determinant, convert it to an MPS, PERFECT-SAMPLE configurations from it, keep
the determinants those samples name, and run CPMC on the resulting
multi-Slater-determinant trial -- deterministically, with the same seed and the
same propagator as every other run in this notebook. With enough samples the
MSD trial IS psi_T again and the run must reproduce the earlier ones exactly.

WHY THIS IS A REAL TEST. Each sampled configuration |n> is a product state in
the site basis, i.e. a Slater determinant whose orbital matrix is a set of unit
columns. A set of them carrying the exact amplitudes c(n) = <n|psi_T> is psi_T
truncated to the sampled support:

    |psi_MSD> = sum_{n in sampled}  c(n) |n>   ->   |psi_T>   as the support fills

Sampling chooses only WHICH determinants appear; the coefficients are exact.
So the run interrogates one thing and nothing else: does perfect sampling visit
the configurations of psi_T with the right probabilities?

WHICH SUPPORT. psi_T is a spin product, so the two channels are sampled
separately and the determinant set is the PRODUCT of the two sampled channel
sets, n_a_distinct x n_b_distinct determinants. The alternative -- keep only the
(n_a, n_b) PAIRS that were actually drawn -- is the literal reading but it
cannot converge here: the rarest channel configuration has p = 3.2e-6, so the
rarest pair has p = 1e-11 and needs ~5e11 draws. Measured on the pair support:
400000 samples reach 83% of the 4900 pairs, fidelity 0.9992, and still 1.2%
error on the overlap. The product support needs only every channel
configuration once: 1.5e6 draws give the rarest one with 99% probability, and
sampling is cheap enough (0.09 us each) that this costs well under a second.

WHY IT STAYS FAST. With a product support the MSD trial factorises exactly the
way psi_T does, so the 4900-determinant expansion never has to be summed as
4900 terms. With w_sigma(n) = c_sigma(n) det(C_sigma[occ(n), :]),

    <psi_MSD|phi> = (sum_a w_a) (sum_b w_b)

and, because the bra of each term is a product state -- so <n|n_i|phi> =
n_i <n|phi>, no Green's function needed for the density --

    E_loc = sum_a w_a k1_a / sum_a w_a  +  (b)  +  U sum_i <n_ia> <n_ib>

with k1 = tr(h1[occ, :] C (C[occ, :])^-1) the one-body mixed element and
<n_i,sigma> = sum_a w_a n_i / sum_a w_a. That is 70 + 70 determinants per
walker instead of 4900. Measured over 100 walkers: overlap 2.2 ms against
82.6 ms, energy 3.4 ms against 159.0 ms -- 38x and 47x. The general
determinant-by-determinant kernels are kept below only to check the factorised
ones against, and they agree to 3e-15.
"""
N_SAMPLES = 2_000_000          # <-- the knob. 1.5e6 is where the trial becomes exact
N_SMALL   = 300                # a deliberately starved trial, for contrast
SAMPLE_KEY = jax.random.PRNGKey(0)
CHUNK = 250_000                # samples per batch, to bound memory


def sample_channel_set(ts, n_samples, key, chunk=CHUNK):
    """Distinct configurations of one channel, with exact amplitudes and hit counts.

    Drawn in batches so n_samples can be large without holding every draw; only
    the distinct configurations survive a batch. Returns (occ, c, counts, n_seen)
    with `occ` the occupied SITE INDICES of each determinant, (ndets, nocc).
    """
    can, _ = right_canonicalize(ts)
    n_sites = len(ts)
    pw = (1 << np.arange(n_sites))[::-1]
    seen = {}
    left, k = int(n_samples), key
    while left > 0:
        k, sub = jax.random.split(k)
        m = min(chunk, left); left -= m
        s = jax.jit(perfect_sample_batch, static_argnums=2)(can, sub, m)
        cfg, amp = np.asarray(s.config), np.asarray(s.amp)
        code = cfg @ pw
        u, first, cnt = np.unique(code, return_index=True, return_counts=True)
        for c, i, n in zip(u, first, cnt):
            if c in seen: seen[c][2] += int(n)
            else:         seen[c] = [cfg[i], float(amp[i]), int(n)]
    keys = sorted(seen)
    occ = np.array([np.where(seen[c][0] == 1)[0] for c in keys])
    return (jnp.asarray(occ), jnp.asarray([seen[c][1] for c in keys]),
            np.array([seen[c][2] for c in keys]), int(n_samples))


def make_msd_kernels(occ_a, ca_, occ_b, cb_, h1, u_int):
    """(overlap, energy) for the product-support MSD trial, in the kernel signature."""
    h1 = jnp.asarray(h1)
    n_sites = h1.shape[0]
    rows = lambda occ: h1[occ]                                    # (nd, nocc, norb)
    hot = lambda occ: jnp.zeros((occ.shape[0], n_sites)).at[
        jnp.arange(occ.shape[0])[:, None], occ].set(1.0)          # (nd, norb)
    ha, hb, oa, ob = rows(occ_a), rows(occ_b), hot(occ_a), hot(occ_b)

    def chan(C, occ, c, hrow):
        M = C[occ]                                                # (nd, nocc, nocc)
        w = c * jnp.linalg.det(M)
        X = jnp.swapaxes(jnp.linalg.solve(
            jnp.swapaxes(M, -1, -2),
            jnp.broadcast_to(C.T, (occ.shape[0],) + C.T.shape)), -1, -2)
        return w, jnp.einsum("kln,knl->k", hrow, X)               # weights, k1 per det

    def overlap(walker, trial_data=None):
        wa = ca_ * jnp.linalg.det(walker[0][occ_a])
        wb = cb_ * jnp.linalg.det(walker[1][occ_b])
        return jnp.sum(wa) * jnp.sum(wb)

    def energy(walker, ham_data=None, meas_ctx=None, trial_data=None):
        wa, k1a = chan(walker[0], occ_a, ca_, ha)
        wb, k1b = chan(walker[1], occ_b, cb_, hb)
        sa, sb = jnp.sum(wa), jnp.sum(wb)
        return (jnp.sum(wa*k1a)/sa + jnp.sum(wb*k1b)/sb
                + u_int * jnp.sum((wa @ oa) * (wb @ ob)) / (sa*sb))

    return overlap, energy


def msd_kernels_general(occ_a, occ_b, coeff, h1, u_int):
    """The same trial summed determinant by determinant, for ANY support. The check
    on the factorised kernels above, and the only option for a non-product support."""
    h1 = jnp.asarray(h1); nd = occ_a.shape[0]; n_sites = h1.shape[0]
    ha, hb = h1[occ_a], h1[occ_b]
    hot = lambda occ: jnp.zeros((nd, n_sites)).at[jnp.arange(nd)[:, None], occ].set(1.0)
    docc = jnp.sum(hot(occ_a) * hot(occ_b), axis=1)

    def solve_x(C, M):
        return jnp.swapaxes(jnp.linalg.solve(
            jnp.swapaxes(M, -1, -2),
            jnp.broadcast_to(C.T, (nd,) + C.T.shape)), -1, -2)

    def overlap(walker, trial_data=None):
        return jnp.sum(coeff * jnp.linalg.det(walker[0][occ_a])
                             * jnp.linalg.det(walker[1][occ_b]))

    def energy(walker, ham_data=None, meas_ctx=None, trial_data=None):
        ca, cb = walker
        Ma, Mb = ca[occ_a], cb[occ_b]
        w = coeff * jnp.linalg.det(Ma) * jnp.linalg.det(Mb)
        k1 = (jnp.einsum("kln,knl->k", ha, solve_x(ca, Ma))
              + jnp.einsum("kln,knl->k", hb, solve_x(cb, Mb)))
        return jnp.sum(w * (k1 + u_int*docc)) / jnp.sum(w)

    return overlap, energy


# ----------------------------------------------------- build the sampled trial
_t0 = time.perf_counter()
occ_a_s, c_a_s, hits_a, _ = sample_channel_set(trial_a, N_SAMPLES, SAMPLE_KEY)
occ_b_s, c_b_s, hits_b, _ = sample_channel_set(trial_b, N_SAMPLES,
                                               jax.random.split(SAMPLE_KEY)[1])
_t_sample = time.perf_counter() - _t0
_n_full = int(round(np.prod([(L - i)/(i + 1) for i in range(n_up)])))   # C(L, n_up)
overlap_msd, energy_msd = make_msd_kernels(occ_a_s, c_a_s, occ_b_s, c_b_s, h1, U)

print(f"{N_SAMPLES} samples per channel in {_t_sample:.1f} s")
print(f"  distinct configurations : alpha {len(c_a_s)}/{_n_full}   beta {len(c_b_s)}/{_n_full}")
print(f"  determinants in the MSD : {len(c_a_s)*len(c_b_s)}")
print(f"  rarest kept, p_hat      : {hits_a.min()/N_SAMPLES:.2e}  (exact min |c|^2 "
      f"{float(jnp.min(c_a_s**2)):.2e})")
print(f"  captured weight sum|c|^2: alpha {float(jnp.sum(c_a_s**2)):.12f}  "
      f"beta {float(jnp.sum(c_b_s**2)):.12f}")

# --------------------------------------------------------------- self-checks
_rng = np.random.default_rng(11)
_W = (jnp.asarray(np.stack([Ca + 0.25*_rng.standard_normal(Ca.shape) for _ in range(32)])),
      jnp.asarray(np.stack([Cb + 0.25*_rng.standard_normal(Cb.shape) for _ in range(32)])))
_ov_ref = np.asarray(jax.vmap(uhf_overlap_u, in_axes=(0, None))(_W, trial_data))
_e_ref = np.asarray(jax.jit(jax.vmap(energy_uhf))(_W))
_ov = np.asarray(jax.jit(jax.vmap(overlap_msd))(_W))
_e = np.asarray(jax.jit(jax.vmap(energy_msd))(_W))
print(f"\nagainst the exact determinant trial, 32 perturbed walkers:")
print(f"  max |<psi_MSD|phi> - <psi_T|phi>| / |<psi_T|phi>| = "
      f"{np.abs(_ov/_ov_ref - 1).max():.2e}")
print(f"  max |E_MSD - E_determinant|                      = {np.abs(_e-_e_ref).max():.2e}")

# the factorised kernels against the plain determinant-by-determinant sum
_ia, _ib = [x.ravel() for x in np.meshgrid(np.arange(len(c_a_s)), np.arange(len(c_b_s)),
                                           indexing="ij")]
_ovg, _eg = msd_kernels_general(occ_a_s[_ia], occ_b_s[_ib], c_a_s[_ia]*c_b_s[_ib], h1, U)
_ov2 = np.asarray(jax.jit(jax.vmap(_ovg))(_W)); _e2 = np.asarray(jax.jit(jax.vmap(_eg))(_W))
print(f"  factorised vs {len(_ia)}-determinant sum: overlap "
      f"{np.abs(_ov2/_ov - 1).max():.1e}   energy {np.abs(_e2-_e).max():.1e}")

# ------------------------------------------- how the trial converges with N
def _trial_quality(ns, key):
    oa, ca__, _, _ = sample_channel_set(trial_a, ns, key)
    ob, cb__, _, _ = sample_channel_set(trial_b, ns, jax.random.split(key)[1])
    ov_f, en_f = make_msd_kernels(oa, ca__, ob, cb__, h1, U)
    ov = np.asarray(jax.jit(jax.vmap(ov_f))(_W)); en = np.asarray(jax.jit(jax.vmap(en_f))(_W))
    fid = float(jnp.sum(ca__**2) * jnp.sum(cb__**2))      # <psi_MSD|psi_T>^2/<psi_MSD|psi_MSD>
    return len(ca__), len(cb__), fid, np.abs(ov/_ov_ref - 1).max(), np.abs(en - _e_ref).max()

print(f"\n{'N per channel':>14s} {'configs a':>10s} {'b':>4s} {'dets':>7s} {'fidelity':>12s} "
      f"{'max overlap err':>16s} {'max E err':>11s}")
for _ns in (100, 300, 1000, 10_000, 100_000, 1_000_000):
    _q = _trial_quality(_ns, SAMPLE_KEY)
    print(f"{_ns:14d} {_q[0]:10d} {_q[1]:4d} {_q[0]*_q[1]:7d} {_q[2]:12.8f} "
          f"{_q[3]:16.2e} {_q[4]:11.2e}")

# ------------------------------------------------------------------- the runs
prop_ops = prop_ops_slow
mean_msd, err_msd, be_msd, bw_msd = run(overlap_msd, energy_msd)
be_msd = np.asarray(be_msd)

_oa, _ca, _, _ = sample_channel_set(trial_a, N_SMALL, SAMPLE_KEY)
_ob, _cb, _, _ = sample_channel_set(trial_b, N_SMALL, jax.random.split(SAMPLE_KEY)[1])
_ovs, _ens = make_msd_kernels(_oa, _ca, _ob, _cb, h1, U)
mean_sml, err_sml, be_sml, _ = run(_ovs, _ens)

print(f"\nE_exact (diagonalisation)              {e_exact:.12f}")
print(f"CPMC, MSD from {N_SAMPLES} samples  {float(mean_msd):.12f} +- {float(err_msd):.2e}")
print(f"CPMC, all-determinant                  {float(mean_ref):.12f} +- {float(err_ref):.2e}")
print(f"CPMC, all-MPS                          {float(mean_mps):.12f} +- {float(err_mps):.2e}")
print(f"CPMC, MSD from {N_SMALL} samples "
      f"({len(_ca)}x{len(_cb)} dets)  {float(mean_sml):.12f} +- {float(err_sml):.2e}")
_d = np.abs(be_msd - be_ref)
print(f"\nmean difference, MSD vs determinant   {abs(float(mean_msd)-float(mean_ref)):.3e}")
print(f"|E_msd - E_det| per block: max {_d.max():.3e}   median {np.median(_d):.3e}")
print(f"mean difference, starved MSD          "
      f"{abs(float(mean_sml)-float(mean_ref)):.3e}  <- the trial is genuinely truncated")

2000000 samples per channel in 1.1 s
  distinct configurations : alpha 70/70   beta 70/70
  determinants in the MSD : 4900
  rarest kept, p_hat      : 2.50e-06  (exact min |c|^2 3.18e-06)
  captured weight sum|c|^2: alpha 1.000000000000  beta 1.000000000000

against the exact determinant trial, 32 perturbed walkers:
  max |<psi_MSD|phi> - <psi_T|phi>| / |<psi_T|phi>| = 1.55e-15
  max |E_MSD - E_determinant|                      = 7.99e-15
  factorised vs 4900-determinant sum: overlap 5.6e-16   energy 3.3e-15

 N per channel  configs a    b    dets     fidelity  max overlap err   max E err
           100         38   35    1330   0.73151306         5.44e-01    1.54e+00
           300         52   48    2496   0.93588082         3.65e-01    1.30e+00
          1000         57   58    3306   0.99076120         8.32e-02    3.24e-01
         10000         68   66    4488   0.99954305         9.21e-03    3.17e-02
        100000         69   69    4761   0.99999364         1.47e-03    7.12e-03